# 配置&导入数据

In [ ]:
# ==================== 单元格 1: 导入库 ====================
"""
TFT异常检测和变点检测主流程
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import train_test_split


plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
from tqdm.auto import tqdm
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import CountVectorizer
# 引入你现有的配置和预处理模块

# from utilis_preprocess import tokenize_zh_factory
import jieba

In [ ]:

# ==================== 单元格 2: 加载数据 ====================
"""
加载模拟数据、运营日历、真实数据
"""
import os
# 数据路径（根据实际情况修改）

DATA_PATH = r'C:\tongji\0 code\00_data\03_TFT'
RAW_DATA_PATH = r'C:\tongji\0 code\00_data\01_EDA'
DETECTION_DATA_PATH = r'C:\tongji\0 code\03_core_model_v3\output'

# 加载数据
print("📂 加载数据...")


df_official_real = pd.read_csv(RAW_DATA_PATH +os.sep+ 'temp_all_official_standart_time_type.csv')# 真实官方运营日历
df_features = pd.read_csv(RAW_DATA_PATH +os.sep+ 'Time_Series_Features_2024-12-05 00 to 2025-11-27 00_15min.csv')# 真实数据集
df_raw = pd.read_csv(RAW_DATA_PATH +os.sep+ 'temp_all_standart_time_2cleaned.csv')# 真实原始数据
df_detected = pd.read_csv(DETECTION_DATA_PATH +os.sep+ 'adaptive_detection_resultv5.csv')# 变点检测结果

In [ ]:
crisis_df =  pd.read_csv(r'C:\tongji\0 code\00_data\01_EDA\0_crisis_event'+os.sep+'crisis_event_pool.csv')


In [ ]:
df_detected['timestamp'] = pd.to_datetime(df_detected['timestamp'])
df_features.rename(columns={'Unnamed: 0': 'timestamp'}, inplace=True)
df_features['timestamp'] = pd.to_datetime(df_features['timestamp'])
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'])

In [ ]:
# 导入真实官方事件
df_official_real['event'] = df_official_real['official_author']+' ' + df_official_real['category']
# df_official_real.drop(columns=['official_author','category'], inplace=True)
df_official_real = df_official_real[['event','timestamp']]
df_official_real['timestamp'] = pd.to_datetime(df_official_real['timestamp'])

In [ ]:
# df_detected['pred_label'].value_counts()


In [ ]:
df_raw.info()

In [ ]:
df_features.info()

# 提取异常时间窗口

## 提取原始事件

In [ ]:
import pandas as pd

def filter_raw_data_by_anomaly_windows(df1, df2):
    # 1. 根据条件筛选 df1 的数据
    condition = (df1['is_baseline_drift'] == True) | (df1['pred_label'].isin(['CP', 'AP']))
    df1_filtered = df1[condition].copy()

    # 2. 将时间列转换为 datetime 对象，并将无效时间转为 NaT
    df1_filtered['attribution_time'] = pd.to_datetime(df1_filtered['attribution_time'], errors='coerce')
    df2['timestamp'] = pd.to_datetime(df2['timestamp'], errors='coerce')

    # 提取所有有效且不为空的归因时间点，转换为列表
    valid_times = df1_filtered['attribution_time'].dropna().tolist()

    # 3. 初始化 df2 的全局布尔掩码（长度与 df2 相同，初始全为 False）
    final_mask = pd.Series(False, index=df2.index)

    # 4. 遍历每一个有效的时间点，计算落入窗口的掩码
    for end_time in valid_times:
        # 定义 15 分钟的时间窗口边界
        start_time = end_time - pd.Timedelta(minutes=15)
        
        # 找出 df2 中时间落入当前窗口的行 (左开右闭)
        current_mask = (df2['timestamp'] > start_time) & (df2['timestamp'] <= end_time)
        
        # 使用位运算符 "|" (OR) 将当前窗口的掩码叠加到全局掩码中
        # 这可以自动处理多个时间窗口重叠的情况，避免数据重复提取
        final_mask = final_mask | current_mask

    # 5. 使用最终生成的全局掩码，提取 df2 中的原始数据
    df2_filtered = df2[final_mask].copy()

    return df2_filtered

# 使用示例：


In [ ]:
filtered_raw_df = filter_raw_data_by_anomaly_windows(df_detected, df_raw)
print(f"成功筛选出 {len(filtered_raw_df)} 条相关数据。")

In [ ]:
filtered_raw_df.columns

In [ ]:
filtered_raw_df_output = filtered_raw_df[filtered_raw_df['sentiment_score']<0.5].copy()
filtered_raw_df_output.shape

In [ ]:
filtered_raw_df_output[['timestamp','post_type', 'text_clean_strict', 'label', 'sentiment_score', 'is_negative', 'img_phashes',
       'has_img', 'img_count']][:5000].to_csv('filtered_raw_data-1.csv', index=False, encoding='utf-8-sig')

# 原始事件+主题建模

### 新版词云

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def set_chinese_font():
    """设置中文字体，解决方块问题"""
    fonts = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'SimSun', 'PingFang SC']
    plt.rcParams['font.sans-serif'] = fonts + plt.rcParams['font.sans-serif']
    plt.rcParams['axes.unicode_minus'] = False

def generate_academic_business_dashboard(topic_summary, window_id, title_info, output_path):
    """
    生成适用于运营与论文双重场景的归因仪表盘（修复字体与脏数据问题）。
    """
    if not topic_summary:
        print("无主题汇总数据，无法生成看板。")
        return

    # 1. 强制应用全局字体设置
    set_chinese_font()
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 7.5), dpi=300)
    fig.suptitle(f"实时归因预警与主题分布分析 | {title_info}", fontsize=20, fontweight='bold', color='#333333', y=1.02)

    # ==========================================
    # 模块 1：左侧 - 主题分布条形图 (处理脏数据)
    # ==========================================
    ax_bar = axes[0]
    
    # 使用 .copy() 避免修改原始数据内存
    raw_topic_counts = topic_summary.get('top_seed_topics', {}).copy()
    discovered = topic_summary.get('discovered_clusters', {})
    
    # 【核心修复】清除上一版代码残留在字典里的英文 "New Cluster" 脏数据
    keys_to_remove = [k for k in raw_topic_counts.keys() if 'Cluster' in k or '未知' in k]
    for k in keys_to_remove:
        del raw_topic_counts[k]
    
    # 中文映射字典
    topic_translation = {
        'Unknown': '未知主题',
        "Strategy_Capacity":"战略规划" ,
        "Story_Narrative": "剧情",
        'World_Design': '大世界设计',
        "Gameplay_Modes": " 游戏玩法",
        "UX_Controls": "基础体验",
        "Visual_Arts": "视觉表现",
        'Tech_Quality': '技术质量',
        'Progression_Res': '数值与养成体验',
        "Gacha_Quality":"商品质量",
        "Monetization_Price":"商业化定价",
        'Ops_Marketing': '宣发与运营',

    }
    
    display_counts = {}
    for k, v in raw_topic_counts.items():
        display_counts[topic_translation.get(k, k)] = v
        
    for cid, info in discovered.items():
        display_counts[f"未知主题 {cid}"] = info['count']
        
    if not display_counts:
        ax_bar.text(0.5, 0.5, "暂无明确主题特征", ha='center', fontsize=16)
        ax_bar.axis('off')
    else:
        # 排序并绘制
        sorted_topics = sorted(display_counts.items(), key=lambda x: x[1], reverse=False)
        labels = [item[0] for item in sorted_topics]
        counts = [item[1] for item in sorted_topics]
        
        sns.barplot(x=counts, y=labels, ax=ax_bar, palette='Blues_r')
        
        total_samples = topic_summary.get('total_samples', 0)
        ax_bar.set_title(f"预警窗口负面主题分布 (总样本量: {total_samples})", fontsize=15, pad=15)
        ax_bar.set_xlabel("负面样本数量", fontsize=12)
        ax_bar.set_ylabel("归因主题分类", fontsize=12)
        
        ax_bar.xaxis.grid(True, linestyle='--', alpha=0.6)
        ax_bar.set_axisbelow(True)
        
        for i, v in enumerate(counts):
            ax_bar.text(v + 0.3, i, str(v), color='#333333', va='center', fontweight='bold', fontsize=11)

    # ==========================================
    # 模块 2：右侧 - 核心关键词与摘要卡片 (修复字体支持)
    # ==========================================
    ax_text = axes[1]
    ax_text.axis('off') 
    
    report_text = "【核心突发特征提取】\n\n"
    
    if discovered:
        for cid, info in discovered.items():
            report_text += f"发现异常主题 [未知主题 {cid}] (样本数: {info['count']})\n"
            report_text += f"核心高频特征词: {', '.join(info['keywords'][:8])}\n"
            report_text += "-" * 45 + "\n"
    else:
        report_text += "当前窗口未发现新聚集的未知异常主题。\n\n"
        
    # report_text += "\n【归因研判与处置建议】\n"
    # report_text += "1. 重点关注包含上述“核心高频特征词”的异常反馈明细。\n"
    # report_text += "2. 结合左侧“主题分布”最高项，排查对应业务与技术系统。\n"
    # report_text += "3. 若该报警被确认为中期冲击，建议立即启动相关应急预案。"

    # 【核心修复】删除了 family='monospace' 参数，使其默认继承设置好的全局中文字体
    ax_text.text(0.05, 0.9, report_text, fontsize=13, va='top', linespacing=2.0,
                 bbox=dict(facecolor='#f8f9fa', edgecolor='#cccccc', boxstyle='square,pad=1.5', alpha=0.8))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ 学术业务双用仪表盘已生成: {output_path}")

# 使用示例：
# generate_academic_business_dashboard(
#     topic_summary, 
#     window_id="202504282345", 
#     title_info="04-28 23:45 异常告警", 
#     output_path="./academic_business_dashboard.png"
# )

### V2

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import textwrap  # 引入文本换行模块

def set_chinese_font():
    fonts = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'SimSun', 'PingFang SC']
    plt.rcParams['font.sans-serif'] = fonts + plt.rcParams['font.sans-serif']
    plt.rcParams['axes.unicode_minus'] = False

def generate_academic_business_dashboard(topic_summary, window_id, title_info, output_path):
    if not topic_summary:
        return

    set_chinese_font()
    
    # 【修改】增大画布的高度和宽度，从 (15, 7.5) 调整为 (18, 11) 以容纳完整长文本
    fig, axes = plt.subplots(1, 2, figsize=(18, 11), dpi=300)
    fig.suptitle(f"实时归因预警与定性分析 | {title_info}", fontsize=22, fontweight='bold', color='#333333', y=1.02)

    # ================== 左侧：条形图 ==================
    ax_bar = axes[0]
    raw_topic_counts = topic_summary.get('top_seed_topics', {}).copy()
    discovered = topic_summary.get('discovered_clusters', {})
    
    keys_to_remove = [k for k in raw_topic_counts.keys() if 'Cluster' in k or '未知' in k]
    for k in keys_to_remove:
        del raw_topic_counts[k]
    
    topic_translation = {
        'Unknown': '未知主题',
        "Strategy_Capacity":"战略规划" ,
        "Story_Narrative": "剧情",
        'World_Design': '大世界设计',
        "Gameplay_Modes": " 游戏玩法",
        "UX_Controls": "基础体验",
        "Visual_Arts": "视觉表现",
        'Tech_Quality': '技术质量',
        'Progression_Res': '数值与养成体验',
        "Gacha_Quality":"商品质量",
        "Monetization_Price":"商业化定价",
        'Ops_Marketing': '宣发与运营',

    }
    
    display_counts = {}
    reverse_mapping = {} 
    
    for k, v in raw_topic_counts.items():
        cn_name = topic_translation.get(k, k)
        display_counts[cn_name] = v
        reverse_mapping[cn_name] = k
        
    for cid, info in discovered.items():
        cn_name = f"突发聚集簇 {cid}"
        display_counts[cn_name] = info['count']
        reverse_mapping[cn_name] = f"Cluster_{cid}"

    if not display_counts:
        ax_bar.text(0.5, 0.5, "暂无明确主题特征", ha='center', fontsize=16)
        ax_bar.axis('off')
    else:
        sorted_topics = sorted(display_counts.items(), key=lambda x: x[1], reverse=False)
        labels = [item[0] for item in sorted_topics]
        counts = [item[1] for item in sorted_topics]
        
        sns.barplot(x=counts, y=labels, ax=ax_bar, palette='Blues_r')
        total_samples = topic_summary.get('total_samples', 0)
        ax_bar.set_title(f"预警窗口高危主题分布 (总样本量: {total_samples})", fontsize=16, pad=15)
        ax_bar.set_xlabel("异常样本数量", fontsize=13)
        ax_bar.set_ylabel("归因主题分类", fontsize=13)
        ax_bar.xaxis.grid(True, linestyle='--', alpha=0.6)
        ax_bar.set_axisbelow(True)
        
        for i, v in enumerate(counts):
            ax_bar.text(v + 0.3, i, str(v), color='#333333', va='center', fontweight='bold', fontsize=12)

    # ================== 右侧：定性文本抽样 (完整换行显示) ==================
    ax_text = axes[1]
    ax_text.axis('off') 
    
    rep_texts = topic_summary.get('representative_texts', {})
    report_text = "【核心高频反馈内容抽样 (按长度优先)】\n\n"
    
    top_topics_desc = sorted_topics[::-1][:4]
    
    has_text = False
    for display_name, _ in top_topics_desc:
        original_key = reverse_mapping.get(display_name)
        if original_key == 'Unknown': 
            original_key = 'Unknown_Scatter'
            
        texts = rep_texts.get(original_key, [])
        if texts:
            has_text = True
            report_text += f"■ {display_name}:\n"
            for idx, text in enumerate(texts):
                # 【核心修改】使用 textwrap 进行自动换行
                # width=42 表示每行约 42 个中文字符
                # subsequent_indent 保证换行后的文本能与第一行的文本对齐缩进
                wrapped_text = textwrap.fill(
                    text, 
                    width=42, 
                    initial_indent=f"  {idx+1}. ", 
                    subsequent_indent="     "
                )
                report_text += f"{wrapped_text}\n"
            report_text += "\n"
            
    if not has_text:
        report_text += "当前过滤条件下，无足够长度的有效文本样本。\n"

    # va='top' 确保文本从顶部开始对齐
    ax_text.text(0.02, 0.98, report_text, fontsize=12, va='top', linespacing=1.6,
                 bbox=dict(facecolor='#f8f9fa', edgecolor='#cccccc', boxstyle='square,pad=1.5', alpha=0.9))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✅ 学术业务双用仪表盘已生成: {output_path}")
    plt.close()

### 整体调用

In [ ]:
final_reasoning_df = process_reasoning_and_modeling(df_reasoning, filtered_raw_df)


In [ ]:
final_reasoning_df[final_reasoning_df['sentiment_score']<0.5].columns

In [ ]:
final_reasoning_df[final_reasoning_df['sentiment_score']<0.5][['timestamp','post_type', 'text_clean_strict', 'label', 'sentiment_score', 'is_negative', 'img_phashes',
       'has_img', 'img_count', 'topic_modeling_result']][:5000].to_csv('final_reasoning_results-1.csv', index=False, encoding='utf-8-sig')

### 单独调用

In [ ]:
# import pandas as pd
# import importlib
# import os
# import jieba
# import re
# import importlib
# # 1. 强制重载修改后的 Python 文件
# importlib.reload(version_config)
# importlib.reload(utilis_preprocess)
# from utilis_preprocess import tokenize_zh_factory

# # 2. 重新实例化引擎（确保它使用的是刚加载的最新类）

# # ==========================================
# # 1. 动态更新配置 (无需重启 Notebook)
# # ==========================================
# # 重新加载停用词配置
# import version_config
# importlib.reload(version_config)
# NIKKI_STOPWORDS = set(version_config.NIKKI_STOPWORDS)
# print("成功重新加载 version_config.py，已获取最新停用词！")
# SEED_TOPICS = version_config.SEED_TOPICS
# # 重新加载自定义专有名词字典
# dict_path = 'nikki_dict.txt'
# if os.path.exists(dict_path):
#     jieba.load_userdict(dict_path)
#     print(f"成功重新加载 {dict_path}，已获取最新专有名词！")
# else:
#     print(f"提示：未找到字典文件 {dict_path}")


# importlib.reload(TFT_topic_modeling)
# importlib.reload(TFT_Attribution_engine)
# print("✅ 模块热加载成功，已更新提取原贴逻辑！")
# from TFT_Attribution_engine import ContentAttributionEngine 

# # ==========================================
# # 2. 准备数据与初始化引擎
# # ==========================================
# engine = ContentAttributionEngine()

# target_time_str = "2025-04-28 23:45:00" 
# end_time = pd.to_datetime(target_time_str)
# start_time = end_time - pd.Timedelta(minutes=15)

# filtered_raw_df['timestamp'] = pd.to_datetime(filtered_raw_df['timestamp'], errors='coerce')
# window_mask = (filtered_raw_df['timestamp'] > start_time) & (filtered_raw_df['timestamp'] <= end_time)
# df_window = filtered_raw_df[window_mask].copy()

# # 情感过滤
# df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

# if not df_window_negative.empty:
#     # ==========================================
#     # 3. 核心新增：查看被送入模型的干净文本长什么样
#     # ==========================================
#     print(f"\n========== 【数据检查】窗口内负向文本共计 {len(df_window_negative)} 条 ==========")
    
#     # 确定引擎内部最终使用的是哪一列
#     text_col = 'text_clean_strict' if 'text_clean_strict' in df_window_negative.columns else 'text_clean'
    
#     # 去除空值后，提取前 10 条进行打印对照
#     check_df = df_window_negative.dropna(subset=[text_col])
    
#     count = 1
#     for idx, row in check_df.iterrows():
#         print(f"[{count}] 原贴内容 : {row['text']}")
#         print(f"    清洗结果 : {row[text_col]}")
#         print("-" * 50)
#         count += 1
#     # 1. 实例化分词工厂（包含词性过滤和停用词过滤）
#     # 假设你在 tokenize_zh_factory 内部已经写好了 n 和 v 的词性过滤逻辑
#     tokenizer = tokenize_zh_factory(set(NIKKI_STOPWORDS))

#     def deep_clean_text(text):
#         if not isinstance(text, str) or text.strip() == '':
#             return ""
            
#         # a. 正则去除特殊字符和表情，只保留中英文和数字
#         text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
        
#         # b. 使用分词器进行分词、停用词过滤和词性筛选
#         tokens = tokenizer(text)
        
#         # c. 核心新增：过滤掉纯数字，并进行“句内词汇去重” (保持原有语序)
#         seen = set()
#         unique_tokens = []
        
#         for w in tokens:
#             # 过滤掉纯数字组成的字符串 (比如 "180", "15", "220")
#             # 如果你希望保留数字，可以把 `and not w.isdigit()` 删掉
#             if w not in seen and not w.isdigit():
#                 seen.add(w)
#                 unique_tokens.append(w)
                
#         # d. 重新拼接
#         return " ".join(unique_tokens)
#     # 3. 对 dataframe 应用深度清洗
#     print("\n-> 正在执行深度文本清洗...")
#     df_window_negative['text_clean_strict_deep'] = df_window_negative['text_clean_strict'].apply(deep_clean_text)

#     # 4. 覆盖原来的列，确保 analyze_text 使用的是深度清洗后的文本
#     # df_window_negative['text_clean_strict'] = df_window_negative['text_clean_strict_deep']

#     # 可选：打印几个深度清洗后的结果看看效果
#     print("\n--- 深度清洗后的文本示例 ---")
#     for text in df_window_negative['text_clean_strict_deep'].dropna():
#         print(text)
#     print("----------------------------")       
#     # ==========================================
#     # 4. 运行主题建模并输出结果
#     # ==========================================
#     print("\n-> 正在基于更新后的配置运行主题建模...")
#     window_id = end_time.strftime("%Y%m%d%H%M")
#     title_info = f"{end_time.strftime('%m-%d %H:%M')} [Negative Topics Test]"
    
#     topic_summary = engine.analyze_text(df_window_negative, window_id=window_id, title_info=title_info)
    
#     print("\n========== 【最终结果】主题建模输出 ==========")
#     print(topic_summary)
# else:
#     print(f"提示：该窗口 ({start_time} 至 {end_time}) 内没有情感分数小于 0.5 的数据。")

### 单独V2

In [ ]:
import pandas as pd
import os
import jieba
import re
import importlib

# ==========================================
# 1. 动态更新配置与核心依赖库 (统一处理，避免重复)
# ==========================================
import version_config
import utilis_preprocess
import TFT_topic_modeling
import TFT_Attribution_engine

importlib.reload(version_config)
importlib.reload(utilis_preprocess)
importlib.reload(TFT_topic_modeling)
importlib.reload(TFT_Attribution_engine)
print("✅ 模块热加载成功！已获取最新停用词、清洗逻辑与提取原贴逻辑。")

# 获取最新配置
NIKKI_STOPWORDS = set(version_config.NIKKI_STOPWORDS)
from utilis_preprocess import tokenize_zh_factory
from TFT_Attribution_engine import ContentAttributionEngine 

# ==========================================
# 2. 重新加载专有名词字典
# ==========================================
dict_path = 'nikki_dict.txt'
if os.path.exists(dict_path):
    jieba.load_userdict(dict_path)
    print(f"✅ 成功加载自定义词典: {dict_path}")
else:
    print(f"⚠️ 提示：未找到字典文件 {dict_path}")

# ==========================================
# 3. 准备数据与初始化引擎
# ==========================================
# 必须在此处实例化引擎，确保其读取的是热加载后的最新模型类
engine = ContentAttributionEngine()

target_time_str = "2025-04-28 23:45:00" 
end_time = pd.to_datetime(target_time_str)
start_time = end_time - pd.Timedelta(minutes=15)

filtered_raw_df['timestamp'] = pd.to_datetime(filtered_raw_df['timestamp'], errors='coerce')
window_mask = (filtered_raw_df['timestamp'] > start_time) & (filtered_raw_df['timestamp'] <= end_time)
df_window = filtered_raw_df[window_mask].copy()

# 情感过滤 (负向)
df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

if not df_window_negative.empty:
    print(f"\n========== 【数据检查】窗口内负向文本共计 {len(df_window_negative)} 条 ==========")
    
    # 确定原始清洗文本列名
    text_col = 'text_clean_strict' if 'text_clean_strict' in df_window_negative.columns else 'text_clean'
    
    # 实例化分词工厂 (依赖 utilis_preprocess.py 中包含 jieba.posseg 词性过滤的最新代码)
    tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)

    def deep_clean_text(text):
        if not isinstance(text, str) or text.strip() == '':
            return ""
            
        # a. 正则去除特殊字符和表情
        text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
        
        # b. 词性与停用词过滤分词
        tokens = tokenizer(text)
        
        # c. 过滤纯数字，进行“句内词汇去重”
        seen = set()
        unique_tokens = []
        for w in tokens:
            if w not in seen and not w.isdigit():
                seen.add(w)
                unique_tokens.append(w)
                
        # d. 重新拼接
        return " ".join(unique_tokens)
        
    # 执行深度清洗，生成专属建模列 text_clean_strict_deep (绝对不要覆盖原列)
    print("-> 正在执行深度文本清洗 (词性过滤 + 骨架去重)...")
    df_window_negative['text_clean_strict_deep'] = df_window_negative[text_col].apply(deep_clean_text)

    # 打印前 3 条深层清洗结果用于验证
    print("\n--- 深度清洗后的文本示例 (Top 3) ---")
    for text in df_window_negative['text_clean_strict_deep'].dropna().head(3):
        print(text)
    print("----------------------------")       
    
    # ==========================================
    # 4. 运行主题建模并输出结果
    # ==========================================
    print("\n-> 正在基于更新后的配置运行主题建模...")
    window_id = end_time.strftime("%Y%m%d%H%M")
    title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
    
    topic_summary = engine.analyze_text(df_window_negative, window_id=window_id, title_info=title_info)
    
    # ==========================================
    # 5. 生成学术业务双用仪表盘
    # ==========================================
    # 假设 generate_academic_business_dashboard 已在上方或另一个 Cell 中定义好
    output_png_path = f"./attribution_results/{window_id}/academic_business_dashboard.png"
    generate_academic_business_dashboard(
        topic_summary, 
        window_id=window_id, 
        title_info=title_info, 
        output_path=output_png_path
    )
    
else:
    print(f"提示：该窗口 ({start_time} 至 {end_time}) 内没有情感分数小于 0.5 的数据。")

In [ ]:
window_id = end_time.strftime("%Y%m%d%H%M")

# 4. 生成包含原文抽样的新版看板
generate_academic_business_dashboard(
    topic_summary, 
    window_id=window_id, 
    title_info="04-28 23:45 异常告警",
    output_path=f"./attribution_results/{window_id}/dashboard.svg"
)

# V0

In [ ]:
# window_id = end_time.strftime("%Y%m%d%H%M")
# generate_academic_business_dashboard(
#     topic_summary, 
#     window_id="202504282345", 
#     title_info="04-28 23:45 异常告警", 
#     output_path="./"+window_id+"business_dashboar1.svg"
# )

# V3可视化

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import textwrap # 核心引入：用于文本自动换行对齐

def set_chinese_font():
    """设置学术与业务通用中文字体"""
    # 严格按照规范列表设置，不使用Emoji字体
    fonts = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'SimSun', 'PingFang SC']
    plt.rcParams['font.sans-serif'] = fonts + plt.rcParams['font.sans-serif']
    plt.rcParams['axes.unicode_minus'] = False

def generate_academic_business_dashboard(topic_summary, window_id, title_info, output_path):
    """
    生成适用于运营与论文双重场景的归因仪表盘。
    【方案3重构版】：展示切分后的完整短句，加入自动换行与对齐，移除强行截断。
    """
    if not topic_summary:
        print("无主题汇总数据，无法生成看板。")
        return

    # 1. 应用字体设置
    set_chinese_font()
    
    # 2. 创建 1x2 画布，使用适合展示多条文本的分辨率
    fig, axes = plt.subplots(1, 2, figsize=(16, 8.5), dpi=300)
    fig.suptitle(f"实时舆情精细归因与内容定性分析 | {title_info}", fontsize=20, fontweight='bold', color='#333333', y=1.02)

    # ==========================================
    # 模块 1：左侧 - 主题分布条形图 (严谨中文标签)
    # ==========================================
    ax_bar = axes[0]
    
    raw_topic_counts = topic_summary.get('top_seed_topics', {}).copy()
    discovered = topic_summary.get('discovered_clusters', {})
    
    # 清除上一版代码残留的英文脏数据，确保纯净
    keys_to_remove = [k for k in raw_topic_counts.keys() if 'Cluster' in k or '未知' in k]
    for k in keys_to_remove:
        del raw_topic_counts[k]
    
    # 学术化中文映射字典
    topic_translation = {
        'Unknown': '未知主题',
        "Strategy_Capacity":"战略规划" ,
        "Story_Narrative": "剧情",
        'World_Design': '大世界设计',
        "Gameplay_Modes": " 游戏玩法",
        "UX_Controls": "基础体验",
        "Visual_Arts": "视觉表现",
        'Tech_Quality': '技术质量',
        'Progression_Res': '数值与养成体验',
        "Gacha_Quality":"商品质量与定价",
        # "Monetization_Price":"商业化定价",
        'Ops_Marketing': '宣发与运营',

    }
    
    # 构建全中文的统计字典，并建立映射关系供右侧使用
    display_counts = {}
    display_to_original_map = {} 
    
    for k, v in raw_topic_counts.items():
        cn_name = topic_translation.get(k, k)
        display_counts[cn_name] = v
        display_to_original_map[cn_name] = k
        
    for cid, info in discovered.items():
        cn_name = f"突发聚集簇 {cid}"
        display_counts[cn_name] = info['count']
        # 注意：这里的键名需要与 TopicModeler 返回的 topic_indices 键名对应
        # 假设 TFT_topic_modeling.py 返回的是 f"Cluster_{c}"
        display_to_original_map[cn_name] = f"Cluster_{cid}"

    if not display_counts:
        ax_bar.text(0.5, 0.5, "当前窗口数据较少\n未能识别明确主题", ha='center', fontsize=16)
        ax_bar.axis('off')
    else:
        # 排序并绘制
        sorted_topics = sorted(display_counts.items(), key=lambda x: x[1], reverse=False)
        labels = [item[0] for item in sorted_topics]
        counts = [item[1] for item in sorted_topics]
        
        # 使用 seaborn 绘制精美条形图
        sns.barplot(x=counts, y=labels, ax=ax_bar, palette='Blues_r')
        
        total_samples = topic_summary.get('total_samples', 0)
        ax_bar.set_title(f"预警窗口负面主题分布 (总短句数: {total_samples})", fontsize=15, pad=15)
        ax_bar.set_xlabel("句子样本数量", fontsize=12)
        ax_bar.set_ylabel("归因主题分类", fontsize=12)
        ax_bar.xaxis.grid(True, linestyle='--', alpha=0.6)
        ax_bar.set_axisbelow(True)
        
        # 在柱子上显示具体数字
        for i, v in enumerate(counts):
            ax_bar.text(v + 0.3, i, str(v), color='#333333', va='center', fontweight='bold', fontsize=11)

    # ==========================================
    # 模块 2：右侧 - TOP主题内容定性抽样 (方案3核心)
    # ==========================================
    ax_text = axes[1]
    ax_text.axis('off') 
    
    # 获取归因引擎准备好的代表性文本
    rep_texts = topic_summary.get('representative_texts', {})
    
    report_text = "【TOP级异常主题核心反馈抽样】\n\n"
    
    # 按照左侧条形图的降序排列，取前 3-4 个主题进行原文展示
    top_topics_desc = sorted_topics[::-1][:4]
    
    has_text = False
    for display_name, _ in top_topics_desc:
        original_key = display_to_original_map.get(display_name)
        # 处理 Unknown 分类的特殊键名
        if original_key == 'Unknown': 
            original_key = 'Unknown_Scatter'
            
        texts = rep_texts.get(original_key, [])
        if texts:
            has_text = True
            report_text += f"■ {display_name}:\n"
            for idx, text in enumerate(texts):
                clean_text = str(text).replace('\n', ' ').strip()
                # ！！！【方案3核心修改】彻底移除 [:55] + "..." 强行截断 ！！！
                
                # 【严谨排版】使用 textwrap 自动换行
                # width 表示每行显示的中文字符数量，保证在卡片内不溢出
                # subsequent_indent 保证换行后的文本能对齐缩进
                wrapped_text = textwrap.fill(clean_text, width=42, initial_indent=f"  {idx+1}. ", subsequent_indent="     ")
                report_text += f"{wrapped_text}\n"
                
            report_text += "\n" # 主题间空行
            
    if not has_text:
        report_text += "当前窗口未提取到具备代表性的有效文本样本。\n"

    # va='top' 确保文本从顶部开始对齐，linespacing 提供学术图表所需的行间距
    ax_text.text(0.05, 0.95, report_text, fontsize=12, va='top', linespacing=1.8,
                 bbox=dict(facecolor='#f8f9fa', edgecolor='#cccccc', boxstyle='square,pad=1.5', alpha=0.8))

    # 4. 调整布局并高分辨率保存
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ 学术业务双用仪表盘 (基于句子级完整显示) 已生成: {output_path}")



In [ ]:
import pandas as pd
import importlib
import os
import jieba
import re
from typing import List

# ==========================================
# 1. 强制热加载所有配置与依赖 (解决 Gacha_Quality 丢失)
# ==========================================
import version_config
import utilis_preprocess
import TFT_topic_modeling
import TFT_Attribution_engine

importlib.reload(version_config)
importlib.reload(utilis_preprocess)
importlib.reload(TFT_topic_modeling)
importlib.reload(TFT_Attribution_engine)
print("✅ 模块与配置热加载成功！已获取最新 SEED_TOPICS。")

NIKKI_STOPWORDS = set(version_config.NIKKI_STOPWORDS)
from utilis_preprocess import tokenize_zh_factory
from TFT_Attribution_engine import ContentAttributionEngine 

# 重新加载自定义字典
dict_path = 'nikki_dict.txt'
if os.path.exists(dict_path):
    jieba.load_userdict(dict_path)

# ==========================================
# 2. 增强版：社交媒体中文句子切分函数
# ==========================================
def split_into_sentences(text: str) -> List[str]:
    if not isinstance(text, str): return []
    text = text.replace('\r\n', ' ').replace('\n', ' ')
    
    # 增强正则：支持中英文标点、省略号、Emoji叹号问号
    pattern = r'(?<=[。！？\!\?…❗❗️❓])'
    sentences = re.split(pattern, text)
    
    clean_sents = []
    for s in sentences:
        # 社交媒体常直接用较长的空格断句，这里做二次切分
        subs = re.split(r'\s{2,}', s) 
        for sub in subs:
            if len(sub.strip()) > 2:
                clean_sents.append(sub.strip())
    return clean_sents

# ==========================================
# 3. 准备数据与句子级展开
# ==========================================
engine = ContentAttributionEngine() # 必须重新实例化以读取新配置

target_time_str = "2025-04-28 23:45:00" 
end_time = pd.to_datetime(target_time_str)
start_time = end_time - pd.Timedelta(minutes=15)

filtered_raw_df['timestamp'] = pd.to_datetime(filtered_raw_df['timestamp'], errors='coerce')
window_mask = (filtered_raw_df['timestamp'] > start_time) & (filtered_raw_df['timestamp'] <= end_time)
df_window = filtered_raw_df[window_mask].copy()
df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

if not df_window_negative.empty:
    print(f"\n========== 【句子级展开】原负向帖子共 {len(df_window_negative)} 条 ==========")
    
    display_text_col = 'text_clean_strict' if 'text_clean_strict' in df_window_negative.columns else 'text_clean'
    
    # 切分句子
    df_window_negative['sentences'] = df_window_negative[display_text_col].apply(split_into_sentences)
    
    # 展开为多行
    df_sentences = df_window_negative.explode('sentences').reset_index().rename(columns={'index': 'original_post_id'})
    df_sentences = df_sentences.dropna(subset=['sentences'])
    
    # ！！！【核心修复】强制覆盖原有的长文本列，防止引擎画图时读错 ！！！
    df_sentences['text_clean_strict'] = df_sentences['sentences']
    df_sentences['text'] = df_sentences['sentences']
    
    print(f"✅ 展开完成，生成独立语义短句共 {len(df_sentences)} 条。")

    # ==========================================
    # 4. 深度文本清洗 (仅限名词/动词，供聚类使用)
    # ==========================================
    tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)

    def deep_clean_text(text):
        if not isinstance(text, str) or text.strip() == '': return ""
        text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
        tokens = tokenizer(text)
        seen = set()
        unique_tokens = []
        for w in tokens:
            if w not in seen and not w.isdigit():
                seen.add(w)
                unique_tokens.append(w)
        return " ".join(unique_tokens)
        
    print("-> 正在执行独立句子的深度文本清洗...")
    df_sentences['text_clean_strict_deep'] = df_sentences['text'].apply(deep_clean_text)
    df_sentences = df_sentences[df_sentences['text_clean_strict_deep'].str.len() > 0].copy()

    # ==========================================
    # 5. 运行归因引擎建模与出图
    # ==========================================
    print("\n-> 正在基于句子级数据运行归因建模...")
    window_id = end_time.strftime("%Y%m%d%H%M")
    title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
    
    topic_summary = engine.analyze_text(df_sentences, window_id=window_id, title_info=title_info)
    
    print("\n========== 【最终结果】主题建模输出 (基于短句) ==========")
    
    # 生成看板图表
    generate_academic_business_dashboard(
        topic_summary, 
        window_id=window_id, 
        title_info=title_info,
        output_path=f"dashboard.png" # 建议用png，svg在某些notebook里渲染有问题
    )
    print("看板生成完毕！")

else:
    print(f"提示：该窗口 ({start_time} 至 {end_time}) 内没有负向数据。")

# V4可视化

In [ ]:
import pandas as pd
import importlib
import os
import jieba
import re
import textwrap
from typing import List

# ==========================================
# 1. 强制热加载所有配置与依赖
# ==========================================
import version_config
import utilis_preprocess
import TFT_topic_modeling
import TFT_Attribution_engine

importlib.reload(version_config)
importlib.reload(utilis_preprocess)
importlib.reload(TFT_topic_modeling)
importlib.reload(TFT_Attribution_engine)
print("✅ 模块与配置热加载成功！")

NIKKI_STOPWORDS = set(version_config.NIKKI_STOPWORDS)
from utilis_preprocess import tokenize_zh_factory
from TFT_Attribution_engine import ContentAttributionEngine 

dict_path = 'nikki_dict.txt'
if os.path.exists(dict_path):
    jieba.load_userdict(dict_path)
def set_chinese_font():
    """设置中文字体，解决方块问题"""
    fonts = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'SimSun', 'PingFang SC']
    plt.rcParams['font.sans-serif'] = fonts + plt.rcParams['font.sans-serif']
    plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 2. 【方案A】重构业务仪表盘生成函数
# ==========================================
def generate_academic_business_dashboard(topic_summary, window_id, title_info, output_path):
    if not topic_summary: return
    set_chinese_font()
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 10.5), dpi=300)
    fig.suptitle(f"实时舆情精细归因与内容定性分析 | {title_info}", fontsize=22, fontweight='bold', color='#333333', y=1.02)

    ax_bar = axes[0]
    raw_topic_counts = topic_summary.get('top_seed_topics', {}).copy()
    discovered = topic_summary.get('discovered_clusters', {})
    
    keys_to_remove = [k for k in raw_topic_counts.keys() if 'Cluster' in k or '未知' in k]
    for k in keys_to_remove: 
        del raw_topic_counts[k]
    
    topic_translation = {
        'Unknown': '未能分类散点',
        "Story_Narrative":'文案',
        "Visual_Arts": '外观',
        'Gameplay_World': '大世界与玩法体验',
        'Tech_UX': '技术优化与交互体验',
        'Gacha_Progression': '抽卡与养成',
        "Pricing_Rights":"定价",
        'Ops_Marketing': '运营宣发与版本规划'
    }
    
    display_counts = {}
    display_to_original_map = {} 
    
    # 映射已知主题
    for k, v in raw_topic_counts.items():
        if v > 0: # 过滤掉数量为 0 的主题
            cn_name = topic_translation.get(k, k)
            display_counts[cn_name] = v
            display_to_original_map[cn_name] = k
        
    # 【核心修复 3】动态命名未分类聚类，移除 Emoji 乱码
    for cid, info in discovered.items():
        kws = info['keywords']
        kw_str = " ".join(kws)
        
        # 提取前两个关键词作为该簇的名字 (例如 "鸟套-池子")
        name_suffix = f"{kws[0]}-{kws[1]}" if len(kws) >= 2 else f"簇 {cid}"
        
        crisis_triggers = ['保底', '抽数', '差评', '退钱', '拆分', '内存', '优化', '作死', '消费者', '权益']
        
        # 移除了前缀的 Emoji，改用严谨的方括号
        if any(w in kw_str for w in crisis_triggers) or info['count'] > 25:
            # cn_name = f"[高危] 综合维权事件 ({name_suffix})"
            cn_name = f"综合事件 ({name_suffix})"

        else:
            cn_name = f"其他 ({name_suffix})"
            
        display_counts[cn_name] = info['count']
        display_to_original_map[cn_name] = f"Cluster_{cid}"

    if not display_counts:
        ax_bar.text(0.5, 0.5, "当前窗口数据较少", ha='center', fontsize=16)
        ax_bar.axis('off')
    else:
        # 降序排列
        sorted_topics = sorted(display_counts.items(), key=lambda x: x[1], reverse=True)
        labels = [item[0] for item in sorted_topics]
        counts = [item[1] for item in sorted_topics]
        
        sns.barplot(x=counts, y=labels, ax=ax_bar, palette='Blues_r')
        total_samples = topic_summary.get('total_samples', 0)
        ax_bar.set_title(f"预警窗口负面主题分布 : {total_samples}条短句", fontsize=15, pad=15)
        ax_bar.set_xlabel("句子样本数量", fontsize=13)
        ax_bar.set_ylabel("归因主题分类", fontsize=13)
        ax_bar.xaxis.grid(True, linestyle='--', alpha=0.6)
        ax_bar.set_axisbelow(True)
        
        for i, v in enumerate(counts):
            ax_bar.text(v + 0.3, i, str(v), color='#333333', va='center', fontweight='bold', fontsize=12)

    # 右侧文本渲染
    ax_text = axes[1]
    ax_text.axis('off') 
    rep_texts = topic_summary.get('representative_texts', {})
    report_text = "【TOP级异常主题核心反馈抽样】\n\n"
    
    top_topics_desc = sorted_topics[:6] 
    
    for display_name, count_val in top_topics_desc:
        original_key = display_to_original_map.get(display_name)
        if original_key == 'Unknown': 
            original_key = 'Unknown_Scatter'
            
        texts = rep_texts.get(original_key, [])
        report_text += f"■ {display_name} (含 {count_val} 句):\n"
        
        if texts:
            for idx, text in enumerate(texts):
                clean_text = str(text).replace('\n', ' ').strip()
                wrapped_text = textwrap.fill(clean_text, width=42, initial_indent=f"  {idx+1}. ", subsequent_indent="     ")
                report_text += f"{wrapped_text}\n"
        else:
            report_text += "  [提示]：该分类多为零散短词或重复情绪宣泄，未抽取出有效长句。\n"
        report_text += "\n"

    ax_text.text(0.02, 0.98, report_text, fontsize=11, va='top', linespacing=1.6,
                 bbox=dict(facecolor='#f8f9fa', edgecolor='#cccccc', boxstyle='square,pad=1.5', alpha=0.9))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    # print(f"✅ 看板已生成: {output_path}")
# ==========================================
# 3. 增强版：社交媒体句子切分 (解决短句不触发)
# ==========================================
def split_into_sentences(text: str) -> List[str]:
    if not isinstance(text, str): return []
    text = text.replace('\r\n', ' ').replace('\n', ' ')
    
    # 增加分号，以及社交媒体常见的各种符号
    pattern = r'(?<=[。！？\!\?…❗❗️❓；;])'
    parts = re.split(pattern, text)
    clean_sents = []
    for p in parts:
        # 【关键增强】：社交媒体玩家喜欢用连续空格分段，强制按两个以上空格切分
        subs = re.split(r'\s{2,}', p)
        for sub in subs:
            sub = sub.strip()
            if len(sub) > 3: # 过滤毫无意义的单字
                clean_sents.append(sub)
    return clean_sents


## 运行

In [ ]:

# ==========================================
# 4. 准备数据、切分、展开与归因
# ==========================================
engine = ContentAttributionEngine() 

# target_time_str = "2025-04-28 23:45:00" # 事件2
target_time_str = "2025-07-06 23:45:00" # 事件8

end_time = pd.to_datetime(target_time_str)
start_time = end_time - pd.Timedelta(minutes=15)

filtered_raw_df['timestamp'] = pd.to_datetime(filtered_raw_df['timestamp'], errors='coerce')
window_mask = (filtered_raw_df['timestamp'] > start_time) & (filtered_raw_df['timestamp'] <= end_time)
df_window = filtered_raw_df[window_mask].copy()
df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

if df_window_negative.empty:
    print("当前窗口无数据。")
else:
    display_text_col = 'text_clean_strict' 
    
    # 1. 句子切分与展开
    df_window_negative['sentences'] = df_window_negative[display_text_col].apply(split_into_sentences)
    
    df_sentences = df_window_negative.explode('sentences').reset_index().rename(columns={'index': 'original_post_id'})
    df_sentences = df_sentences.dropna(subset=['sentences'])
    
    # 强制覆盖原列，确保引擎和画图拿到的都是切碎的短句
    df_sentences['text_clean_strict'] = df_sentences['sentences']
    df_sentences['text'] = df_sentences['sentences']

    # 2. 深度清洗 (过滤供模型用的列)
    tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)
    def deep_clean_text(text):
        if not isinstance(text, str) or text.strip() == '': return ""
        
        # 【核心修复 1】首先干掉微博的中括号表情文本，如 [允悲]、[裂开]
        text = re.sub(r'\[.*?\]', '', text)
        
        # 预处理：保留中英数字
        text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
        
        tokens = tokenizer(text)
        seen = set()
        unique_tokens = []
        for w in tokens:
            if w not in seen and not w.isdigit():
                seen.add(w)
                unique_tokens.append(w)
        return " ".join(unique_tokens)
        
    df_sentences['text_clean_strict_deep'] = df_sentences['text_clean_strict'].apply(deep_clean_text)
    df_sentences = df_sentences[df_sentences['text_clean_strict_deep'].str.len() > 0].copy()

    # 3. 运行模型并出图
    print(f"\n-> 正在基于 {len(df_sentences)} 条短句运行归因建模...")
    window_id = end_time.strftime("%Y%m%d%H%M")
    title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
    
    topic_summary = engine.analyze_text(df_sentences, window_id=window_id, title_info=title_info)
    
    output_path = f"./attribution_results/{window_id}/dashboard.svg"
    generate_academic_business_dashboard(topic_summary, window_id, title_info, output_path)
    print(f"✅ 看板已生成: {output_path}")

In [ ]:
df_window_negative


# 未报警窗口

In [ ]:
import pandas as pd
import importlib
import os
import jieba
import re
import textwrap
from typing import List

# ==========================================
# 1. 强制热加载所有配置与依赖
# ==========================================
import version_config
import utilis_preprocess
import TFT_topic_modeling
import TFT_Attribution_engine

importlib.reload(version_config)
importlib.reload(utilis_preprocess)
importlib.reload(TFT_topic_modeling)
importlib.reload(TFT_Attribution_engine)
print("✅ 模块与配置热加载成功！")

NIKKI_STOPWORDS = set(version_config.NIKKI_STOPWORDS)
from utilis_preprocess import tokenize_zh_factory
from TFT_Attribution_engine import ContentAttributionEngine 

dict_path = 'nikki_dict.txt'
if os.path.exists(dict_path):
    jieba.load_userdict(dict_path)

# ==========================================
# 2. 【方案A】重构业务仪表盘生成函数
# ==========================================
# def generate_academic_business_dashboard(topic_summary, window_id, title_info, output_path):
#     if not topic_summary: return
#     fonts = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'SimSun']
#     import matplotlib.pyplot as plt
#     import seaborn as sns
#     plt.rcParams['font.sans-serif'] = fonts + plt.rcParams['font.sans-serif']
#     plt.rcParams['axes.unicode_minus'] = False
    
#     # 增加画布高度，容纳更多文本
#     fig, axes = plt.subplots(1, 2, figsize=(17, 10), dpi=300)
#     fig.suptitle(f"实时舆情精细归因与内容定性分析 | {title_info}", fontsize=20, fontweight='bold', color='#333333', y=1.02)

#     ax_bar = axes[0]
#     raw_topic_counts = topic_summary.get('top_seed_topics', {}).copy()
#     discovered = topic_summary.get('discovered_clusters', {})
    
#     keys_to_remove = [k for k in raw_topic_counts.keys() if 'Cluster' in k or '未知' in k]
#     for k in keys_to_remove: del raw_topic_counts[k]
    
#     # 【更新】适应你的合并逻辑

#     topic_translation = {
#         'Unknown': '未能分类散点',
#         'Story_Visuals': '剧情与视听表现',
#         'Gameplay_World': '大世界与玩法体验',
#         'Tech_UX': '技术优化与交互体验',
#         'Progression_Res': '数值养成与产出',
#         'Gacha_Monetization': '商品质量与商业化',
#         'Ops_Strategy': '运营宣发与版本规划'
#     }
    
#     display_counts = {}
#     display_to_original_map = {} 
    
#     for k, v in raw_topic_counts.items():
#         cn_name = topic_translation.get(k, k)
#         display_counts[cn_name] = v
#         display_to_original_map[cn_name] = k
        
#     for cid, info in discovered.items():
#         # 【方案A 核心逻辑】：嗅探危机关键词，自动升级预警级别
#         kw_str = " ".join(info['keywords'])
#         # 定义能够触发“综合危机事件”的高危词根
#         crisis_triggers = ['保底', '抽数', '差评', '退钱', '拆分', '内存', '优化', '作死', '消费者', '权益']
        
#         if any(w in kw_str for w in crisis_triggers) or info['count'] > 25:
#             cn_name = f"🚨 1.5版综合危机事件 {cid}"
#         else:
#             cn_name = f"突发聚集簇 {cid}"
            
#         display_counts[cn_name] = info['count']
#         display_to_original_map[cn_name] = f"Cluster_{cid}"

#     if not display_counts:
#         ax_bar.axis('off')
#     else:
#         sorted_topics = sorted(display_counts.items(), key=lambda x: x[1], reverse=False)
#         labels = [item[0] for item in sorted_topics]
#         counts = [item[1] for item in sorted_topics]
#         sns.barplot(x=counts, y=labels, ax=ax_bar, palette='Blues_r')
#         total_samples = topic_summary.get('total_samples', 0)
#         ax_bar.set_title(f"预警窗口负面主题分布 (总句数: {total_samples})", fontsize=15, pad=15)
#         ax_bar.set_xlabel("句子样本数量", fontsize=12)
#         ax_bar.set_ylabel("归因主题分类", fontsize=12)
#         ax_bar.xaxis.grid(True, linestyle='--', alpha=0.6)
#         ax_bar.set_axisbelow(True)
#         for i, v in enumerate(counts):
#             ax_bar.text(v + 0.3, i, str(v), color='#333333', va='center', fontweight='bold', fontsize=11)

#     # 右侧排版
#     ax_text = axes[1]
#     ax_text.axis('off') 
#     rep_texts = topic_summary.get('representative_texts', {})
#     report_text = "【TOP级异常主题核心反馈抽样】\n\n"
    
#     # 取前 5 个分类展示 (之前是4个)
#     top_topics_desc = sorted_topics[::-1][:5]
#     has_text = False
    
#     for display_name, _ in top_topics_desc:
#         original_key = display_to_original_map.get(display_name)
#         if original_key == 'Unknown': original_key = 'Unknown_Scatter'
#         texts = rep_texts.get(original_key, [])
#         if texts:
#             has_text = True
#             report_text += f"■ {display_name}:\n"
#             for idx, text in enumerate(texts):
#                 clean_text = str(text).replace('\n', ' ').strip()
#                 # 自动换行
#                 wrapped_text = textwrap.fill(clean_text, width=45, initial_indent=f"  {idx+1}. ", subsequent_indent="     ")
#                 report_text += f"{wrapped_text}\n"
#             report_text += "\n"
            
#     if not has_text: report_text += "当前窗口无有效文本样本。\n"
#     ax_text.text(0.02, 0.98, report_text, fontsize=11, va='top', linespacing=1.6,
#                  bbox=dict(facecolor='#f8f9fa', edgecolor='#cccccc', boxstyle='square,pad=1', alpha=0.8))
#     plt.tight_layout()
#     plt.savefig(output_path, dpi=300, bbox_inches='tight')
#     plt.close()
def generate_academic_business_dashboard(topic_summary, window_id, title_info, output_path):
    if not topic_summary: return
    set_chinese_font()
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 10.5), dpi=300)
    fig.suptitle(f"实时舆情精细归因与内容定性分析 | {title_info}", fontsize=22, fontweight='bold', color='#333333', y=1.02)

    ax_bar = axes[0]
    raw_topic_counts = topic_summary.get('top_seed_topics', {}).copy()
    discovered = topic_summary.get('discovered_clusters', {})
    
    keys_to_remove = [k for k in raw_topic_counts.keys() if 'Cluster' in k or '未知' in k]
    for k in keys_to_remove: 
        del raw_topic_counts[k]
    
    topic_translation = {
        'Unknown': '未能分类散点',
        "Story_Narrative":'文案',
        "Visual_Arts": '外观',
        'Gameplay_World': '大世界与玩法体验',
        'Tech_UX': '技术优化与交互体验',
        'Gacha_Progression': '抽卡与养成',
        "Pricing_Rights":"定价",
        'Ops_Marketing': '运营宣发与版本规划'
    }
    
    display_counts = {}
    display_to_original_map = {} 
    
    # 映射已知主题
    for k, v in raw_topic_counts.items():
        if v > 0: # 过滤掉数量为 0 的主题
            cn_name = topic_translation.get(k, k)
            display_counts[cn_name] = v
            display_to_original_map[cn_name] = k
        
    # 【核心修复 3】动态命名未分类聚类，移除 Emoji 乱码
    for cid, info in discovered.items():
        kws = info['keywords']
        kw_str = " ".join(kws)
        
        # 提取前两个关键词作为该簇的名字 (例如 "鸟套-池子")
        name_suffix = f"{kws[0]}-{kws[1]}" if len(kws) >= 2 else f"簇 {cid}"
        
        crisis_triggers = ['保底', '抽数', '差评', '退钱', '拆分', '内存', '优化', '作死', '消费者', '权益']
        
        # 移除了前缀的 Emoji，改用严谨的方括号
        if any(w in kw_str for w in crisis_triggers) or info['count'] > 25:
            # cn_name = f"[高危] 综合维权事件 ({name_suffix})"
            cn_name = f"综合事件 ({name_suffix})"

        else:
            cn_name = f"其他 ({name_suffix})"
            
        display_counts[cn_name] = info['count']
        display_to_original_map[cn_name] = f"Cluster_{cid}"

    if not display_counts:
        ax_bar.text(0.5, 0.5, "当前窗口数据较少", ha='center', fontsize=16)
        ax_bar.axis('off')
    else:
        # 降序排列
        sorted_topics = sorted(display_counts.items(), key=lambda x: x[1], reverse=True)
        labels = [item[0] for item in sorted_topics]
        counts = [item[1] for item in sorted_topics]
        
        sns.barplot(x=counts, y=labels, ax=ax_bar, palette='Blues_r')
        total_samples = topic_summary.get('total_samples', 0)
        ax_bar.set_title(f"预警窗口负面主题分布 : {total_samples}条短句", fontsize=15, pad=15)
        ax_bar.set_xlabel("句子样本数量", fontsize=13)
        ax_bar.set_ylabel("归因主题分类", fontsize=13)
        ax_bar.xaxis.grid(True, linestyle='--', alpha=0.6)
        ax_bar.set_axisbelow(True)
        
        for i, v in enumerate(counts):
            ax_bar.text(v + 0.3, i, str(v), color='#333333', va='center', fontweight='bold', fontsize=12)

    # 右侧文本渲染
    ax_text = axes[1]
    ax_text.axis('off') 
    rep_texts = topic_summary.get('representative_texts', {})
    report_text = "【TOP级异常主题核心反馈抽样】\n\n"
    
    top_topics_desc = sorted_topics[:6] 
    
    for display_name, count_val in top_topics_desc:
        original_key = display_to_original_map.get(display_name)
        if original_key == 'Unknown': 
            original_key = 'Unknown_Scatter'
            
        texts = rep_texts.get(original_key, [])
        report_text += f"■ {display_name} (含 {count_val} 句):\n"
        
        if texts:
            for idx, text in enumerate(texts):
                clean_text = str(text).replace('\n', ' ').strip()
                wrapped_text = textwrap.fill(clean_text, width=42, initial_indent=f"  {idx+1}. ", subsequent_indent="     ")
                report_text += f"{wrapped_text}\n"
        else:
            report_text += "  [提示]：该分类多为零散短词或重复情绪宣泄，未抽取出有效长句。\n"
        report_text += "\n"

    ax_text.text(0.02, 0.98, report_text, fontsize=11, va='top', linespacing=1.6,
                 bbox=dict(facecolor='#f8f9fa', edgecolor='#cccccc', boxstyle='square,pad=1.5', alpha=0.9))

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    # print(f"✅ 看板已生成: {output_path}")
# ==========================================
# 3. 增强版：社交媒体句子切分 (解决短句不触发)
# ==========================================
def split_into_sentences(text: str) -> List[str]:
    if not isinstance(text, str): return []
    text = text.replace('\r\n', ' ').replace('\n', ' ')
    
    # 增加分号，以及社交媒体常见的各种符号
    pattern = r'(?<=[。！？\!\?…❗❗️❓；;])'
    parts = re.split(pattern, text)
    clean_sents = []
    for p in parts:
        # 【关键增强】：社交媒体玩家喜欢用连续空格分段，强制按两个以上空格切分
        subs = re.split(r'\s{2,}', p)
        for sub in subs:
            sub = sub.strip()
            if len(sub) > 3: # 过滤毫无意义的单字
                clean_sents.append(sub)
    return clean_sents

# ==========================================
# 4. 准备数据、切分、展开与归因
# ==========================================
engine = ContentAttributionEngine() 

target_time_str = "2025-04-27 22:15:00" 
end_time = pd.to_datetime(target_time_str)
start_time = end_time - pd.Timedelta(minutes=15)

df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'], errors='coerce')
window_mask = (df_raw['timestamp'] > start_time) & (df_raw['timestamp'] <= end_time)
df_window = df_raw[window_mask].copy()
df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

if not df_window_negative.empty:
    display_text_col = 'text_clean_strict' if 'text_clean_strict' in df_window_negative.columns else 'text_clean'
    
    # 1. 句子切分与展开
    df_window_negative['sentences'] = df_window_negative[display_text_col].apply(split_into_sentences)
    df_sentences = df_window_negative.explode('sentences').reset_index().rename(columns={'index': 'original_post_id'})
    df_sentences = df_sentences.dropna(subset=['sentences'])
    
    # 强制覆盖原列，确保引擎和画图拿到的都是切碎的短句
    df_sentences['text_clean_strict'] = df_sentences['sentences']
    df_sentences['text'] = df_sentences['sentences']

    # 2. 深度清洗 (过滤供模型用的列)
    tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)
    def deep_clean_text(text):
        if not isinstance(text, str) or text.strip() == '': return ""
        text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
        tokens = tokenizer(text)
        seen = set()
        unique_tokens = []
        for w in tokens:
            if w not in seen and not w.isdigit():
                seen.add(w)
                unique_tokens.append(w)
        return " ".join(unique_tokens)
        
    df_sentences['text_clean_strict_deep'] = df_sentences['text'].apply(deep_clean_text)
    df_sentences = df_sentences[df_sentences['text_clean_strict_deep'].str.len() > 0].copy()

    # 3. 运行模型并出图
    print(f"\n-> 正在基于 {len(df_sentences)} 条短句运行归因建模...")
    window_id = end_time.strftime("%Y%m%d%H%M")
    title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
    
    topic_summary = engine.analyze_text(df_sentences, window_id=window_id, title_info=title_info)
    
    output_path = f"./attribution_results/{window_id}/dashboard.svg"
    generate_academic_business_dashboard(topic_summary, window_id, title_info, output_path)
    print(f"✅ 看板已生成: {output_path}")
else:
    print("当前窗口无数据。")

# V5

In [ ]:
import pandas as pd
import importlib
import os
import jieba
import re
import textwrap
from typing import List
from PIL import Image

# ==========================================
# 1. 强制热加载所有配置与依赖
# ==========================================
import version_config
import utilis_preprocess
import TFT_topic_modeling
import TFT_Attribution_engine

importlib.reload(version_config)
importlib.reload(utilis_preprocess)
importlib.reload(TFT_topic_modeling)
importlib.reload(TFT_Attribution_engine)
print("✅ 模块与配置热加载成功！")

NIKKI_STOPWORDS = set(version_config.NIKKI_STOPWORDS)
from utilis_preprocess import tokenize_zh_factory
from TFT_Attribution_engine import ContentAttributionEngine 

dict_path = 'nikki_dict.txt'
if os.path.exists(dict_path):
    jieba.load_userdict(dict_path)

def set_chinese_font():
    """设置中文字体，解决方块问题"""
    fonts = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'SimSun', 'PingFang SC']
    plt.rcParams['font.sans-serif'] = fonts + plt.rcParams['font.sans-serif']
    plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 2. 【重构】三栏仪表盘：条形图 + 文本抽样 + Top图片
# ==========================================
# // 替换 V5 中的 generate_academic_business_dashboard 函数

def generate_academic_business_dashboard(topic_summary, window_id, title_info, output_path, top_images=None):
    """
    布局：左侧上下两行(条形图+图片) | 右侧全高文本抽样
    """
    if not topic_summary: return
    set_chinese_font()
    
    has_images = top_images and any(img.get('path') and os.path.exists(img['path']) for img in top_images)
    
    import matplotlib.gridspec as gridspec
    
    if has_images:
        valid_images = [img for img in top_images if img.get('path') and os.path.exists(img['path'])]
        n_imgs = min(len(valid_images), 2)
        
        fig = plt.figure(figsize=(20, 12), dpi=300)
        # 左右两大列：左占 45%，右占 55%（给文字更多空间）
        gs_main = gridspec.GridSpec(1, 2, width_ratios=[0.9, 1.1], figure=fig, wspace=0.08)
        
        # 左列再分上下：条形图占 60%，图片占 40%
        gs_left = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs_main[0], 
                                                    height_ratios=[1.2, 1], hspace=0.25)
        ax_bar = fig.add_subplot(gs_left[0])
        
        # 图片区横排
        gs_imgs = gridspec.GridSpecFromSubplotSpec(1, n_imgs, subplot_spec=gs_left[1], wspace=0.1)
        ax_imgs = [fig.add_subplot(gs_imgs[j]) for j in range(n_imgs)]
        
        # 右列：全高文本
        ax_text = fig.add_subplot(gs_main[1])
    else:
        fig = plt.figure(figsize=(20, 11), dpi=300)
        gs_main = gridspec.GridSpec(1, 2, width_ratios=[0.8, 1.2], figure=fig, wspace=0.08)
        ax_bar = fig.add_subplot(gs_main[0])
        ax_text = fig.add_subplot(gs_main[1])
        ax_imgs = []
        valid_images = []
        n_imgs = 0
    
    fig.suptitle(f"实时舆情精细归因与内容定性分析 | {title_info}", 
                 fontsize=22, fontweight='bold', color='#333333', y=0.98)

    # ============ 左上：主题分布条形图 ============
    raw_topic_counts = topic_summary.get('top_seed_topics', {}).copy()
    discovered = topic_summary.get('discovered_clusters', {})
    
    keys_to_remove = [k for k in raw_topic_counts.keys() if 'Cluster' in k or '未知' in k]
    for k in keys_to_remove: 
        del raw_topic_counts[k]
    
    topic_translation = {
        'Unknown': '未能分类散点',
        "Story_Narrative":'文案',
        "Visual_Arts": '外观',
        'Gameplay_World': '大世界与玩法体验',
        'Tech_UX': '技术优化与交互体验',
        'Gacha_Progression': '抽卡与养成',
        "Pricing_Rights":"定价",
        'Ops_Marketing': '运营宣发与版本规划'
    }
    
    display_counts = {}
    display_to_original_map = {} 
    
    for k, v in raw_topic_counts.items():
        if v > 0:
            cn_name = topic_translation.get(k, k)
            display_counts[cn_name] = v
            display_to_original_map[cn_name] = k
        
    for cid, info in discovered.items():
        kws = info['keywords']
        kw_str = " ".join(kws)
        name_suffix = f"{kws[0]}-{kws[1]}" if len(kws) >= 2 else f"簇 {cid}"
        crisis_triggers = ['保底', '抽数', '差评', '退钱', '拆分',   '作死', '消费者', '权益']
        bug_triggers = ['bug', 'BUG', 'Bug', '闪退', '卡死', '崩溃', '黑屏', '白屏', '内存','优化',
                        '穿模', '掉帧', '报错', '卡bug', '花屏', '闪屏', '死机',
                        '卡顿', '断线', '掉线', '进不去', '打不开', '登不上']
        price_triggers = ['定价', '价格', '氪金', '充钱', '钱', '贵', '便宜', '划算', '不值']
        rep_texts = topic_summary.get('representative_texts', {})
        
        if any(w in kw_str for w in bug_triggers):
            base_name = '技术优化与交互体验'
            if base_name in display_counts:
                display_counts[base_name] += info['count']
                existing_key = display_to_original_map.get(base_name)
                cluster_key = f"Cluster_{cid}"
                existing_texts = rep_texts.get(existing_key, [])
                cluster_texts = rep_texts.get(cluster_key, [])
                rep_texts[existing_key] = (existing_texts + cluster_texts)[:5]
                continue  # 已合并，跳过后续添加
            else:
                cn_name = base_name
        elif any(w in kw_str for w in price_triggers):
            base_name = '定价'
            if base_name in display_counts:
                display_counts[base_name] += info['count']
                existing_key = display_to_original_map.get(base_name)
                cluster_key = f"Cluster_{cid}"
                existing_texts = rep_texts.get(existing_key, [])
                cluster_texts = rep_texts.get(cluster_key, [])
                rep_texts[existing_key] = (existing_texts + cluster_texts)[:5]
                continue
            else:
                cn_name = base_name
        elif any(w in kw_str for w in crisis_triggers) or info['count'] > 25:
            cn_name = f"综合事件 ({name_suffix})"
        else:
            cn_name = f"其他 ({name_suffix})"
        display_counts[cn_name] = info['count']
        display_to_original_map[cn_name] = f"Cluster_{cid}"

    sorted_topics = []
    if not display_counts:
        ax_bar.text(0.5, 0.5, "当前窗口数据较少", ha='center', fontsize=16)
        ax_bar.axis('off')
    else:
        sorted_topics = sorted(display_counts.items(), key=lambda x: x[1], reverse=True)
        labels = [item[0] for item in sorted_topics]
        counts = [item[1] for item in sorted_topics]
        
        sns.barplot(x=counts, y=labels, ax=ax_bar, palette='Blues_r')
        total_samples = topic_summary.get('total_samples', 0)
        ax_bar.set_title(f"负面主题分布 ({total_samples}条短句)", fontsize=14, pad=10)
        ax_bar.set_xlabel("句子样本数量", fontsize=12)
        ax_bar.set_ylabel("")
        ax_bar.xaxis.grid(True, linestyle='--', alpha=0.6)
        ax_bar.set_axisbelow(True)
        for i, v in enumerate(counts):
            ax_bar.text(v + 0.3, i, str(v), color='#333333', va='center', fontweight='bold', fontsize=11)

    # ============ 左下：Top 重复图片 ============
    if has_images and ax_imgs:
        for i, ax_sub in enumerate(ax_imgs):
            img_data = valid_images[i]
            try:
                pil_img = Image.open(img_data['path']).convert('RGB')
                ax_sub.imshow(pil_img)
                ax_sub.set_title(f"高频图 Top{i+1} | 重复{img_data['count']}次", 
                               fontsize=11, color='#C0392B', fontweight='bold', pad=8)
            except Exception as e:
                ax_sub.text(0.5, 0.5, f"加载失败\n{e}", ha='center', va='center', fontsize=10)
            ax_sub.axis('off')

    # ============ 右侧全高：文本抽样 ============
    ax_text.axis('off') 
    rep_texts = topic_summary.get('representative_texts', {})
    report_text = "【TOP级异常主题核心反馈抽样】\n\n"
    
    # 给更多空间后可以展示更多主题
    top_topics_desc = sorted_topics[:8] 
    
    for display_name, count_val in top_topics_desc:
        original_key = display_to_original_map.get(display_name)
        if original_key == 'Unknown': 
            original_key = 'Unknown_Scatter'
        texts = rep_texts.get(original_key, [])
        report_text += f"■ {display_name} (含 {count_val} 句):\n"
        if texts:
            for idx, text in enumerate(texts):
                clean_text = str(text).replace('\n', ' ').strip()
                # 右侧更宽，可以放更多字符
                wrapped_text = textwrap.fill(clean_text, width=50, 
                                            initial_indent=f"  {idx+1}. ", 
                                            subsequent_indent="     ")
                report_text += f"{wrapped_text}\n"
        else:
            report_text += "  [提示]：该分类多为零散短词，未抽取出有效长句。\n"
        report_text += "\n"

    ax_text.text(0.02, 0.98, report_text, fontsize=11, va='top', linespacing=1.55,
                 bbox=dict(facecolor='#f8f9fa', edgecolor='#cccccc', 
                          boxstyle='square,pad=1.2', alpha=0.9))

    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 3. 句子切分函数 (不变)
# ==========================================
def split_into_sentences(text: str) -> List[str]:
    if not isinstance(text, str): return []
    text = text.replace('\r\n', ' ').replace('\n', ' ')
    # 【新增】将 // 也作为切分符号（微博转发分隔符）
    # 先统一把 // 替换为中文句号，让后续正则一并处理
    text = text.replace('//', '。')
    pattern = r'(?<=[。！？\!\?…❗❗️❓；;])'
    parts = re.split(pattern, text)
    clean_sents = []
    for p in parts:
        subs = re.split(r'\s{2,}', p)
        for sub in subs:
            sub = sub.strip()
            if len(sub) > 3:
                clean_sents.append(sub)
    return clean_sents

def clean_repost_official(text: str) -> str:
    """
    清洗微博转发内容：删除 //@官方账号 后面的所有内容。
    例如："我觉得太差了//@暖暖官方:新版本上线啦..." -> "我觉得太差了"
    """
    if not isinstance(text, str):
        return text
    
    # 定义已知的官方账号关键词（可按需扩展）
    official_keywords = [
        '无限暖暖', '无限暖暖搬砖工', '无限暖暖美鸭梨小助手'
    ]
    
    # 匹配 //@xxx: 或 //@xxx 的模式
    # 从第一个 //@官方账号 开始，截断后面所有内容
    for kw in official_keywords:
        # 匹配 //@包含关键词的用户名（用户名可含中英文数字下划线）
        pattern = rf'//\s*@[^@/]*{re.escape(kw)}[^@/]*[:：]?.*$'
        text = re.sub(pattern, '', text, flags=re.DOTALL)
    
    return text.strip()

## 运行

In [ ]:
# ==========================================
# 4. 准备数据、过滤、切分、展开与归因
# ==========================================
engine = ContentAttributionEngine() 

# target_time_str = "2025-04-28 23:45:00" # 事件2 PA
# target_time_str = "2025-05-03 23:00:00" # 事件2 CA
# target_time_str = "2025-05-04 07:45:00" # 事件2 CA

# target_time_str = "2025-07-06 23:45:00" # 事件3
# target_time_str = "2025-08-31 23:45:00" # 事件7
# target_time_str = "2025-09-02 23:45:00" # 事件8
target_time_str = "2025-09-19 12:45:00" # CP
target_time_str = "2025-10-17 12:45:00" # CP






end_time = pd.to_datetime(target_time_str)
start_time = end_time - pd.Timedelta(minutes=15)

filtered_raw_df['timestamp'] = pd.to_datetime(filtered_raw_df['timestamp'], errors='coerce')
window_mask = (filtered_raw_df['timestamp'] > start_time) & (filtered_raw_df['timestamp'] <= end_time)
df_window = filtered_raw_df[window_mask].copy()
df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

if df_window_negative.empty:
    print("当前窗口无数据。")
else:
    # ====== 【新增步骤 A】提取图片信息 (在过滤之前！) ======
    # 先从完整的负面数据中提取 Top2 高频重复图片
    print("-> 正在提取窗口内高频重复图片...")
    top_images = engine.extract_top_images(df_window_negative, top_k=2)
    if top_images:
        for i, img in enumerate(top_images):
            print(f"  Top {i+1}: count={img['count']}, hash={img.get('hash','N/A')}, path={img.get('path','无本地文件')}")
    else:
        print("  未检测到重复传播图片。")
    
    # ====== 【新增步骤 B】过滤噪声帖子 (含图/视频/官方) ======
    df_window_filtered = engine.filter_noise_posts(df_window_negative)
    if 'video_page_url' in df_window_filtered.columns:
        before_count = len(df_window_filtered)
        df_window_filtered = df_window_filtered[
            df_window_filtered['video_page_url'].isna() | 
            (df_window_filtered['video_page_url'].astype(str).str.strip() == '') |
            (df_window_filtered['video_page_url'].astype(str).str.lower() == 'nan')
        ].copy()
        print(f"  已过滤含视频帖子: {before_count - len(df_window_filtered)} 条")
    
    # 【新增】清洗转发官方内容
    display_text_col = 'text_clean_strict'
    df_window_filtered[display_text_col] = df_window_filtered[display_text_col].apply(clean_repost_official)
    # 清洗后可能变空，再次过滤
    df_window_filtered = df_window_filtered[
        df_window_filtered[display_text_col].notna() & 
        (df_window_filtered[display_text_col].str.len() > 3)
    ].copy()
    if df_window_filtered.empty:
        print("⚠️ 过滤后无剩余纯文本帖子，跳过主题建模。")
    else:
        display_text_col = 'text_clean_strict' 
        
        # 1. 句子切分与展开
        df_window_filtered['sentences'] = df_window_filtered[display_text_col].apply(split_into_sentences)
        df_sentences = df_window_filtered.explode('sentences').reset_index().rename(columns={'index': 'original_post_id'})
        df_sentences = df_sentences.dropna(subset=['sentences'])
        
        df_sentences['text_clean_strict'] = df_sentences['sentences']
        df_sentences['text'] = df_sentences['sentences']

        # 2. 深度清洗
        tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)
        def deep_clean_text(text):
            if not isinstance(text, str) or text.strip() == '': return ""
            text = re.sub(r'\[.*?\]', '', text)
            text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
            tokens = tokenizer(text)
            seen = set()
            unique_tokens = []
            for w in tokens:
                if w not in seen and not w.isdigit():
                    seen.add(w)
                    unique_tokens.append(w)
            return " ".join(unique_tokens)
            
        df_sentences['text_clean_strict_deep'] = df_sentences['sentences'].apply(deep_clean_text)
        df_sentences = df_sentences[df_sentences['text_clean_strict_deep'].str.len() > 0].copy()

        # 3. 运行模型并出图 (传入 top_images)
        print(f"\n-> 正在基于 {len(df_sentences)} 条纯文本短句运行归因建模...")
        window_id = end_time.strftime("%Y%m%d%H%M")
        title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
        
        topic_summary = engine.analyze_text(df_sentences, window_id=window_id, title_info=title_info)
        
        output_path = f"./attribution_results/{window_id}/dashboard.svg"
        
        # 【核心】将 top_images 传入看板生成函数
        generate_academic_business_dashboard(
            topic_summary, window_id, title_info, output_path,
            top_images=top_images  # 新增参数
        )
        print(f"✅ 看板已生成: {output_path}")

## 未报警窗口

In [ ]:
# ==========================================
# 4. 准备数据、过滤、切分、展开与归因
# ==========================================
engine = ContentAttributionEngine() 

# target_time_str = "2025-04-28 23:45:00" # 事件2 PA
# target_time_str = "2025-05-03 23:00:00" # 事件2 CA
# target_time_str = "2025-05-04 07:45:00" # 事件2 CA

# target_time_str = "2025-07-06 23:45:00" # 事件3
# target_time_str = "2025-08-31 23:45:00" # 事件7
# target_time_str = "2025-09-02 23:45:00" # 事件8

# target_time_str = "2025-04-16 21:00:00" # 
# target_time_str = "2025-07-17 14:15:00" # 
# target_time_str = "2025-07-18 11:30:00" # 
target_time_str = "2025-10-15 11:15:00" # 






end_time = pd.to_datetime(target_time_str)
start_time = end_time - pd.Timedelta(minutes=15)

df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'], errors='coerce')
window_mask = (df_raw['timestamp'] > start_time) & (df_raw['timestamp'] <= end_time)
df_window = df_raw[window_mask].copy()
df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

if df_window_negative.empty:
    print("当前窗口无数据。")
else:
    # ====== 【新增步骤 A】提取图片信息 (在过滤之前！) ======
    # 先从完整的负面数据中提取 Top2 高频重复图片
    print("-> 正在提取窗口内高频重复图片...")
    top_images = engine.extract_top_images(df_window_negative, top_k=2)
    if top_images:
        for i, img in enumerate(top_images):
            print(f"  Top {i+1}: count={img['count']}, hash={img.get('hash','N/A')}, path={img.get('path','无本地文件')}")
    else:
        print("  未检测到重复传播图片。")
    
    # ====== 【新增步骤 B】过滤噪声帖子 (含图/视频/官方) ======
    df_window_filtered = engine.filter_noise_posts(df_window_negative)
    if 'video_page_url' in df_window_filtered.columns:
        before_count = len(df_window_filtered)
        df_window_filtered = df_window_filtered[
            df_window_filtered['video_page_url'].isna() | 
            (df_window_filtered['video_page_url'].astype(str).str.strip() == '') |
            (df_window_filtered['video_page_url'].astype(str).str.lower() == 'nan')
        ].copy()
        print(f"  已过滤含视频帖子: {before_count - len(df_window_filtered)} 条")
    
    # 【新增】清洗转发官方内容
    display_text_col = 'text_clean_strict'
    df_window_filtered[display_text_col] = df_window_filtered[display_text_col].apply(clean_repost_official)
    # 清洗后可能变空，再次过滤
    df_window_filtered = df_window_filtered[
        df_window_filtered[display_text_col].notna() & 
        (df_window_filtered[display_text_col].str.len() > 3)
    ].copy()
    if df_window_filtered.empty:
        print("⚠️ 过滤后无剩余纯文本帖子，跳过主题建模。")
    else:
        display_text_col = 'text_clean_strict' 
        
        # 1. 句子切分与展开
        df_window_filtered['sentences'] = df_window_filtered[display_text_col].apply(split_into_sentences)
        df_sentences = df_window_filtered.explode('sentences').reset_index().rename(columns={'index': 'original_post_id'})
        df_sentences = df_sentences.dropna(subset=['sentences'])
        
        df_sentences['text_clean_strict'] = df_sentences['sentences']
        df_sentences['text'] = df_sentences['sentences']

        # 2. 深度清洗
        tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)
        def deep_clean_text(text):
            if not isinstance(text, str) or text.strip() == '': return ""
            text = re.sub(r'\[.*?\]', '', text)
            text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
            tokens = tokenizer(text)
            seen = set()
            unique_tokens = []
            for w in tokens:
                if w not in seen and not w.isdigit():
                    seen.add(w)
                    unique_tokens.append(w)
            return " ".join(unique_tokens)
            
        df_sentences['text_clean_strict_deep'] = df_sentences['sentences'].apply(deep_clean_text)
        df_sentences = df_sentences[df_sentences['text_clean_strict_deep'].str.len() > 0].copy()

        # 3. 运行模型并出图 (传入 top_images)
        print(f"\n-> 正在基于 {len(df_sentences)} 条纯文本短句运行归因建模...")
        window_id = end_time.strftime("%Y%m%d%H%M")
        title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
        
        topic_summary = engine.analyze_text(df_sentences, window_id=window_id, title_info=title_info)
        
        output_path = f"./attribution_results/{window_id}/dashboard.svg"
        
        # 【核心】将 top_images 传入看板生成函数
        generate_academic_business_dashboard(
            topic_summary, window_id, title_info, output_path,
            top_images=top_images  # 新增参数
        )
        print(f"✅ 看板已生成: {output_path}")

## 整体运行CA

In [ ]:
df_detected_PA  = df_detected[df_detected['pred_label'] =="AP"]
df_detected_CA  = df_detected[df_detected['pred_label'] =="CP"]


In [ ]:
df_detected_PA

In [ ]:
# ==========================================
# 4. 准备数据、过滤、切分、展开与归因
# ==========================================
engine = ContentAttributionEngine() 

# target_time_str = "2025-04-28 23:45:00" # 事件2 PA
# target_time_str = "2025-05-03 23:00:00" # 事件2 CA
# target_time_str = "2025-05-04 07:45:00" # 事件2 CA

# target_time_str = "2025-07-06 23:45:00" # 事件3
# target_time_str = "2025-08-31 23:45:00" # 事件7
# target_time_str = "2025-09-02 23:45:00" # 事件8

# target_time_str = "2025-04-16 21:00:00" # 
# target_time_str = "2025-07-17 14:15:00" # 
# target_time_str = "2025-07-18 11:30:00" # 
from tqdm import tqdm
# for target_time_str in tqdm(df_detected_CA['timestamp']):
for target_time_str in tqdm(df_detected_CA['attribution_time']):

    print(f"\n=== 处理时间点: {target_time_str} ===")
    end_time = pd.to_datetime(target_time_str)
    start_time = end_time - pd.Timedelta(minutes=15)

    df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'], errors='coerce')
    window_mask = (df_raw['timestamp'] > start_time) & (df_raw['timestamp'] <= end_time)
    df_window = df_raw[window_mask].copy()
    df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

    if df_window_negative.empty:
        print("当前窗口无数据。")
    else:
        # ====== 【新增步骤 A】提取图片信息 (在过滤之前！) ======
        # 先从完整的负面数据中提取 Top2 高频重复图片
        print("-> 正在提取窗口内高频重复图片...")
        top_images = engine.extract_top_images(df_window_negative, top_k=2)
        if top_images:
            for i, img in enumerate(top_images):
                print(f"  Top {i+1}: count={img['count']}, hash={img.get('hash','N/A')}, path={img.get('path','无本地文件')}")
        else:
            print("  未检测到重复传播图片。")
        
        # ====== 【新增步骤 B】过滤噪声帖子 (含图/视频/官方) ======
        df_window_filtered = engine.filter_noise_posts(df_window_negative)
        if 'video_page_url' in df_window_filtered.columns:
            before_count = len(df_window_filtered)
            df_window_filtered = df_window_filtered[
                df_window_filtered['video_page_url'].isna() | 
                (df_window_filtered['video_page_url'].astype(str).str.strip() == '') |
                (df_window_filtered['video_page_url'].astype(str).str.lower() == 'nan')
            ].copy()
            print(f"  已过滤含视频帖子: {before_count - len(df_window_filtered)} 条")
        
        # 【新增】清洗转发官方内容
        display_text_col = 'text_clean_strict'
        df_window_filtered[display_text_col] = df_window_filtered[display_text_col].apply(clean_repost_official)
        # 清洗后可能变空，再次过滤
        df_window_filtered = df_window_filtered[
            df_window_filtered[display_text_col].notna() & 
            (df_window_filtered[display_text_col].str.len() > 3)
        ].copy()
        if df_window_filtered.empty:
            print("⚠️ 过滤后无剩余纯文本帖子，跳过主题建模。")
        else:
            display_text_col = 'text_clean_strict' 
            
            # 1. 句子切分与展开
            df_window_filtered['sentences'] = df_window_filtered[display_text_col].apply(split_into_sentences)
            df_sentences = df_window_filtered.explode('sentences').reset_index().rename(columns={'index': 'original_post_id'})
            df_sentences = df_sentences.dropna(subset=['sentences'])
            
            df_sentences['text_clean_strict'] = df_sentences['sentences']
            df_sentences['text'] = df_sentences['sentences']

            # 2. 深度清洗
            tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)
            def deep_clean_text(text):
                if not isinstance(text, str) or text.strip() == '': return ""
                text = re.sub(r'\[.*?\]', '', text)
                text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
                tokens = tokenizer(text)
                seen = set()
                unique_tokens = []
                for w in tokens:
                    if w not in seen and not w.isdigit():
                        seen.add(w)
                        unique_tokens.append(w)
                return " ".join(unique_tokens)
                
            df_sentences['text_clean_strict_deep'] = df_sentences['sentences'].apply(deep_clean_text)
            df_sentences = df_sentences[df_sentences['text_clean_strict_deep'].str.len() > 0].copy()

            # 3. 运行模型并出图 (传入 top_images)
            print(f"\n-> 正在基于 {len(df_sentences)} 条纯文本短句运行归因建模...")
            window_id = end_time.strftime("%Y%m%d%H%M")
            title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
            
            topic_summary = engine.analyze_text(df_sentences, window_id=window_id, title_info=title_info)
            
            output_path = f"./attribution_results/{window_id}/dashboard.svg"
            
            # 【核心】将 top_images 传入看板生成函数
            generate_academic_business_dashboard(
                topic_summary, window_id, title_info, output_path,
                top_images=top_images  # 新增参数
            )
            print(f"✅ 看板已生成: {output_path}")

## 整体运行 PA

In [ ]:
from alive_progress import alive_bar
engine = ContentAttributionEngine() 
from tqdm import tqdm

    # ==========================================
# 4. 准备数据、过滤、切分、展开与归因
# ==========================================
for target_time_str in tqdm(df_detected_PA['timestamp'][:350],
                            desc="总体执行进度",
                            position=0, 
                            leave=True,
                            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'):
    print(f"\n=== 处理时间点: {target_time_str} ===")
    end_time = pd.to_datetime(target_time_str)
    start_time = end_time - pd.Timedelta(minutes=15)

    df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'], errors='coerce')
    window_mask = (df_raw['timestamp'] > start_time) & (df_raw['timestamp'] <= end_time)
    df_window = df_raw[window_mask].copy()
    df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

    if df_window_negative.empty:
        print("当前窗口无数据。")
    else:
        # ====== 【新增步骤 A】提取图片信息 (在过滤之前！) ======
        # 先从完整的负面数据中提取 Top2 高频重复图片
        print("-> 正在提取窗口内高频重复图片...")
        top_images = engine.extract_top_images(df_window_negative, top_k=2)
        if top_images:
            for i, img in enumerate(top_images):
                print(f"  Top {i+1}: count={img['count']}, hash={img.get('hash','N/A')}, path={img.get('path','无本地文件')}")
        else:
            print("  未检测到重复传播图片。")
        
        # ====== 【新增步骤 B】过滤噪声帖子 (含图/视频/官方) ======
        df_window_filtered = engine.filter_noise_posts(df_window_negative)
        if 'video_page_url' in df_window_filtered.columns:
            before_count = len(df_window_filtered)
            df_window_filtered = df_window_filtered[
                df_window_filtered['video_page_url'].isna() | 
                (df_window_filtered['video_page_url'].astype(str).str.strip() == '') |
                (df_window_filtered['video_page_url'].astype(str).str.lower() == 'nan')
            ].copy()
            print(f"  已过滤含视频帖子: {before_count - len(df_window_filtered)} 条")
        
        # 【新增】清洗转发官方内容
        display_text_col = 'text_clean_strict'
        df_window_filtered[display_text_col] = df_window_filtered[display_text_col].apply(clean_repost_official)
        # 清洗后可能变空，再次过滤
        df_window_filtered = df_window_filtered[
            df_window_filtered[display_text_col].notna() & 
            (df_window_filtered[display_text_col].str.len() > 3)
        ].copy()
        if df_window_filtered.empty:
            print("⚠️ 过滤后无剩余纯文本帖子，跳过主题建模。")
        else:
            display_text_col = 'text_clean_strict' 
            
            # 1. 句子切分与展开
            df_window_filtered['sentences'] = df_window_filtered[display_text_col].apply(split_into_sentences)
            df_sentences = df_window_filtered.explode('sentences').reset_index().rename(columns={'index': 'original_post_id'})
            df_sentences = df_sentences.dropna(subset=['sentences'])
            
            df_sentences['text_clean_strict'] = df_sentences['sentences']
            df_sentences['text'] = df_sentences['sentences']

            # 2. 深度清洗
            tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)
            def deep_clean_text(text):
                if not isinstance(text, str) or text.strip() == '': return ""
                text = re.sub(r'\[.*?\]', '', text)
                text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
                tokens = tokenizer(text)
                seen = set()
                unique_tokens = []
                for w in tokens:
                    if w not in seen and not w.isdigit():
                        seen.add(w)
                        unique_tokens.append(w)
                return " ".join(unique_tokens)
                
            df_sentences['text_clean_strict_deep'] = df_sentences['sentences'].apply(deep_clean_text)
            df_sentences = df_sentences[df_sentences['text_clean_strict_deep'].str.len() > 0].copy()

            # 3. 运行模型并出图 (传入 top_images)
            print(f"\n-> 正在基于 {len(df_sentences)} 条纯文本短句运行归因建模...")
            window_id = end_time.strftime("%Y%m%d%H%M")
            title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
            
            topic_summary = engine.analyze_text(df_sentences, window_id=window_id, title_info=title_info)
            
            output_path = f"./attribution_results/{window_id}/dashboard.svg"
            
            # 【核心】将 top_images 传入看板生成函数
            generate_academic_business_dashboard(
                topic_summary, window_id, title_info, output_path,
                top_images=top_images  # 新增参数
            )
            print(f"✅ 看板已生成: {output_path}")

## 整体运行 PA2

In [ ]:
from alive_progress import alive_bar
engine = ContentAttributionEngine() 
from tqdm import tqdm

    # ==========================================
# 4. 准备数据、过滤、切分、展开与归因
# ==========================================
for target_time_str in tqdm(df_detected_PA['timestamp'][350:],
                            desc="总体执行进度",
                            position=0, 
                            leave=True,
                            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'):
    print(f"\n=== 处理时间点: {target_time_str} ===")
    end_time = pd.to_datetime(target_time_str)
    start_time = end_time - pd.Timedelta(minutes=15)

    df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'], errors='coerce')
    window_mask = (df_raw['timestamp'] > start_time) & (df_raw['timestamp'] <= end_time)
    df_window = df_raw[window_mask].copy()
    df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()

    if df_window_negative.empty:
        print("当前窗口无数据。")
    else:
        # ====== 【新增步骤 A】提取图片信息 (在过滤之前！) ======
        # 先从完整的负面数据中提取 Top2 高频重复图片
        print("-> 正在提取窗口内高频重复图片...")
        top_images = engine.extract_top_images(df_window_negative, top_k=2)
        if top_images:
            for i, img in enumerate(top_images):
                print(f"  Top {i+1}: count={img['count']}, hash={img.get('hash','N/A')}, path={img.get('path','无本地文件')}")
        else:
            print("  未检测到重复传播图片。")
        
        # ====== 【新增步骤 B】过滤噪声帖子 (含图/视频/官方) ======
        df_window_filtered = engine.filter_noise_posts(df_window_negative)
        if 'video_page_url' in df_window_filtered.columns:
            before_count = len(df_window_filtered)
            df_window_filtered = df_window_filtered[
                df_window_filtered['video_page_url'].isna() | 
                (df_window_filtered['video_page_url'].astype(str).str.strip() == '') |
                (df_window_filtered['video_page_url'].astype(str).str.lower() == 'nan')
            ].copy()
            print(f"  已过滤含视频帖子: {before_count - len(df_window_filtered)} 条")
        
        # 【新增】清洗转发官方内容
        display_text_col = 'text_clean_strict'
        df_window_filtered[display_text_col] = df_window_filtered[display_text_col].apply(clean_repost_official)
        # 清洗后可能变空，再次过滤
        df_window_filtered = df_window_filtered[
            df_window_filtered[display_text_col].notna() & 
            (df_window_filtered[display_text_col].str.len() > 3)
        ].copy()
        if df_window_filtered.empty:
            print("⚠️ 过滤后无剩余纯文本帖子，跳过主题建模。")
        else:
            display_text_col = 'text_clean_strict' 
            
            # 1. 句子切分与展开
            df_window_filtered['sentences'] = df_window_filtered[display_text_col].apply(split_into_sentences)
            df_sentences = df_window_filtered.explode('sentences').reset_index().rename(columns={'index': 'original_post_id'})
            df_sentences = df_sentences.dropna(subset=['sentences'])
            
            df_sentences['text_clean_strict'] = df_sentences['sentences']
            df_sentences['text'] = df_sentences['sentences']

            # 2. 深度清洗
            tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)
            def deep_clean_text(text):
                if not isinstance(text, str) or text.strip() == '': return ""
                text = re.sub(r'\[.*?\]', '', text)
                text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
                tokens = tokenizer(text)
                seen = set()
                unique_tokens = []
                for w in tokens:
                    if w not in seen and not w.isdigit():
                        seen.add(w)
                        unique_tokens.append(w)
                return " ".join(unique_tokens)
                
            df_sentences['text_clean_strict_deep'] = df_sentences['sentences'].apply(deep_clean_text)
            df_sentences = df_sentences[df_sentences['text_clean_strict_deep'].str.len() > 0].copy()

            # 3. 运行模型并出图 (传入 top_images)
            print(f"\n-> 正在基于 {len(df_sentences)} 条纯文本短句运行归因建模...")
            window_id = end_time.strftime("%Y%m%d%H%M")
            title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
            
            topic_summary = engine.analyze_text(df_sentences, window_id=window_id, title_info=title_info)
            
            output_path = f"./attribution_results/{window_id}/dashboard.svg"
            
            # 【核心】将 top_images 传入看板生成函数
            generate_academic_business_dashboard(
                topic_summary, window_id, title_info, output_path,
                top_images=top_images  # 新增参数
            )
            # print(f"✅ 看板已生成: {output_path}")

In [ ]:
mask = (pd.to_datetime(df_raw['timestamp']) > pd.to_datetime('2025-10-25 0:00:00')) & (pd.to_datetime(df_raw['timestamp']) <= pd.to_datetime('2025-11-5 0:15:00'))& (df_raw['sentiment_score'] < 0.5)
df_raw[mask].to_csv("filtered_CA_data.csv", index=False, encoding='utf-8-sig')

## CA单点测试

In [ ]:
# ==========================================
# 4. 准备数据、过滤、切分、展开与归因
# ==========================================
engine = ContentAttributionEngine() 

# target_time_str = "2025-04-28 23:45:00" # 事件2 PA
# target_time_str = "2025-05-03 23:00:00" # 事件2 CA
# target_time_str = "2025-05-04 07:45:00" # 事件2 CA

# target_time_str = "2025-07-06 23:45:00" # 事件3
# target_time_str = "2025-08-31 23:45:00" # 事件7
# target_time_str = "2025-09-02 23:45:00" # 事件8
# target_time_str = "2025-09-19 12:45:00" # CP
# target_time_str = "2025-10-17 12:45:00" # CP

# start_time = pd.to_datetime("2025/8/3  13:45:00")
# end_time = pd.to_datetime("2025/8/4  8:00:00")


# start_time = pd.to_datetime("2025/10/25 11:00")
# end_time = pd.to_datetime("2025/10/25 23:45")


# start_time = pd.to_datetime("2025/10/28 1:15")
# end_time = pd.to_datetime("2025/10/28 21:15")

start_time = pd.to_datetime("2025/11/4 19:45")
end_time = pd.to_datetime("2025/11/5 0:15")

# end_time = pd.to_datetime(target_time_str)
# start_time = end_time - pd.Timedelta(minutes=15)

# filtered_raw_df['timestamp'] = pd.to_datetime(filtered_raw_df['timestamp'], errors='coerce')
# window_mask = (filtered_raw_df['timestamp'] > start_time) & (filtered_raw_df['timestamp'] <= end_time)
# df_window = filtered_raw_df[window_mask].copy()
# df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()
df_raw['timestamp'] = pd.to_datetime(df_raw['timestamp'], errors='coerce')
window_mask = (df_raw['timestamp'] > start_time) & (df_raw['timestamp'] <= end_time)
df_window = df_raw[window_mask].copy()
df_window_negative = df_window[df_window['sentiment_score'] < 0.5].copy()
if df_window_negative.empty:
    print("当前窗口无数据。")
else:
    # ====== 【新增步骤 A】提取图片信息 (在过滤之前！) ======
    # 先从完整的负面数据中提取 Top2 高频重复图片
    print("-> 正在提取窗口内高频重复图片...")
    top_images = engine.extract_top_images(df_window_negative, top_k=2)
    if top_images:
        for i, img in enumerate(top_images):
            print(f"  Top {i+1}: count={img['count']}, hash={img.get('hash','N/A')}, path={img.get('path','无本地文件')}")
    else:
        print("  未检测到重复传播图片。")
    
    # ====== 【新增步骤 B】过滤噪声帖子 (含图/视频/官方) ======
    df_window_filtered = engine.filter_noise_posts(df_window_negative)
    if 'video_page_url' in df_window_filtered.columns:
        before_count = len(df_window_filtered)
        df_window_filtered = df_window_filtered[
            df_window_filtered['video_page_url'].isna() | 
            (df_window_filtered['video_page_url'].astype(str).str.strip() == '') |
            (df_window_filtered['video_page_url'].astype(str).str.lower() == 'nan')
        ].copy()
        print(f"  已过滤含视频帖子: {before_count - len(df_window_filtered)} 条")
    
    # 【新增】清洗转发官方内容
    display_text_col = 'text_clean_strict'
    df_window_filtered[display_text_col] = df_window_filtered[display_text_col].apply(clean_repost_official)
    # 清洗后可能变空，再次过滤
    df_window_filtered = df_window_filtered[
        df_window_filtered[display_text_col].notna() & 
        (df_window_filtered[display_text_col].str.len() > 3)
    ].copy()
    if df_window_filtered.empty:
        print("⚠️ 过滤后无剩余纯文本帖子，跳过主题建模。")
    else:
        display_text_col = 'text_clean_strict' 
        
        # 1. 句子切分与展开
        df_window_filtered['sentences'] = df_window_filtered[display_text_col].apply(split_into_sentences)
        df_sentences = df_window_filtered.explode('sentences').reset_index().rename(columns={'index': 'original_post_id'})
        df_sentences = df_sentences.dropna(subset=['sentences'])
        
        df_sentences['text_clean_strict'] = df_sentences['sentences']
        df_sentences['text'] = df_sentences['sentences']

        # 2. 深度清洗
        tokenizer = tokenize_zh_factory(NIKKI_STOPWORDS)
        def deep_clean_text(text):
            if not isinstance(text, str) or text.strip() == '': return ""
            text = re.sub(r'\[.*?\]', '', text)
            text = re.sub(r'[^\w\s\u4e00-\u9fa5a-zA-Z0-9]', ' ', text)
            tokens = tokenizer(text)
            seen = set()
            unique_tokens = []
            for w in tokens:
                if w not in seen and not w.isdigit():
                    seen.add(w)
                    unique_tokens.append(w)
            return " ".join(unique_tokens)
            
        df_sentences['text_clean_strict_deep'] = df_sentences['sentences'].apply(deep_clean_text)
        df_sentences = df_sentences[df_sentences['text_clean_strict_deep'].str.len() > 0].copy()

        # 3. 运行模型并出图 (传入 top_images)
        print(f"\n-> 正在基于 {len(df_sentences)} 条纯文本短句运行归因建模...")
        window_id = end_time.strftime("%Y%m%d%H%M")
        title_info = f"{end_time.strftime('%m-%d %H:%M')} 异常告警"
        
        topic_summary = engine.analyze_text(df_sentences, window_id=window_id, title_info=title_info)
        
        output_path = f"./attribution_results/{window_id}/dashboard.svg"
        
        # 【核心】将 top_images 传入看板生成函数
        generate_academic_business_dashboard(
            topic_summary, window_id, title_info, output_path,
            top_images=top_images  # 新增参数
        )
        print(f"✅ 看板已生成: {output_path}")

# 文本处理

## 提取合并内容

In [ ]:
import pandas as pd
from collections import Counter

def process_anomalies_and_merge(df1, df2):
    # 1. 筛选 df1 的数据
    condition = (df1['is_baseline_drift'] == True) | (df1['pred_label'].isin(['CP', 'AP']))
    df1_filtered = df1[condition].copy()

    # 2. 修改 pred_label，将 CP 改为 CA，AP 改为 PA
    df1_filtered['pred_label'] = df1_filtered['pred_label'].replace({'CP': 'CA', 'AP': 'PA'})

    # 3. 将时间列转换为 datetime 对象
    # coerce 参数会将无法转换的无效值直接变为 NaT (缺失值)
    df1_filtered['attribution_time'] = pd.to_datetime(df1_filtered['attribution_time'], errors='coerce')
    df2['timestamp'] = pd.to_datetime(df2['timestamp'], errors='coerce')

    # 辅助函数：提取字符串格式的 img_phashes (严格遵循不使用 try except 的要求)
    def clean_and_split_phashes(val):
        if not isinstance(val, str):
            return []
        
        val = val.strip()
        # 如果是字符串形式的列表，例如 "['hash1', 'hash2']"，则去除首尾括号
        if val.startswith('[') and val.endswith(']'):
            val = val[1:-1]
        
        # 移除单引号和双引号
        val = val.replace("'", "").replace('"', "")
        
        # 按照逗号切分，并去除多余空格
        return [h.strip() for h in val.split(',') if h.strip()]

    # 4. 核心逻辑：定义针对每一行的 15 分钟窗口处理函数
    def process_15min_window(row):
        end_time = row['attribution_time']
        
        # 检查时间是否为空
        if pd.isna(end_time):
            return pd.Series({
                'text_concat': '', 
                'top_img_phash': None, 
                'top_img_count': 0, 
                'top_img_ratio': 0.0
            })

        # 定义 15 分钟的窗口（假设区间为: [attribution_time - 15分钟, attribution_time]）
        start_time = end_time - pd.Timedelta(minutes=15)

        # 在 df2 中获取窗口内的数据
        window_df = df2[(df2['timestamp'] > start_time) & (df2['timestamp'] <= end_time)]

        # 5. 拼接 text_clean_strict
        # 排除空值，转换为字符串并使用 " | " 作为分隔符进行拼接
        text_series = window_df['text_clean_strict']
        # 加入清洗
        clean_mask = (
            text_series.notna() & 
            (text_series != 'FILTER_NEUTRAL_LEACHING') & 
            (text_series.str.len() > 1) &
            (~text_series.str.contains(r'^\s*$', regex=True, na=False))
        )
        # 根据掩码提取有效文本，转换为字符串并拼接
        # valid_texts = window_df['text_clean_strict'].dropna().astype(str).tolist()
        valid_texts = text_series[clean_mask].astype(str).tolist()
        text_concat = " | ".join(valid_texts)

        # 6. 统计 img_phashes
        all_phashes = []
        for val in window_df['img_phashes'].dropna():
            if isinstance(val, list):
                # 如果已经是列表类型，直接追加
                all_phashes.extend([str(item) for item in val])
            else:
                # 否则作为字符串进行处理
                all_phashes.extend(clean_and_split_phashes(val))

        # 计算最常出现的 phash 及其占比
        top_img_phash = None
        top_img_count = 0
        top_img_ratio = 0.0

        if len(all_phashes) > 0:
            total_count = len(all_phashes)
            counter = Counter(all_phashes)
            
            # most_common(1) 返回格式如 [('hash_abc', 5)]
            most_common_item = counter.most_common(1)[0]
            top_img_phash = most_common_item[0]
            top_img_count = most_common_item[1]
            top_img_ratio = top_img_count / total_count

        return pd.Series({
            'text_concat': text_concat,
            'top_img_phash': top_img_phash,
            'top_img_count': top_img_count,
            'top_img_ratio': top_img_ratio
        })

    # 应用处理函数并将生成的新列与原始 DataFrame 合并
    window_results = df1_filtered.apply(process_15min_window, axis=1)
    df1_final = pd.concat([df1_filtered, window_results], axis=1)

    return df1_final


In [ ]:
df_reasoning = process_anomalies_and_merge(df_detected, df_raw)



## 辅助函数

In [ ]:
from version_config import SEED_TOPICS, NIKKI_STOPWORDS

from jieba import posseg as pseg
def load_dynamic_stopwords(custom_file_path='vocabulary_frequencies.csv'):
    """
    从外部文件动态加载人工标记的停用词，并与原有的 NIKKI_STOPWORDS 合并。
    """
    # 1. 基础停用词集合初始化
    final_stopwords = set(NIKKI_STOPWORDS)
    
    # 2. 检查文件是否存在
    if not os.path.exists(custom_file_path):
        print(f"提示：未找到文件 {custom_file_path}，将仅使用基础配置。")
        return final_stopwords

    # 3. 读取 CSV 文件
    df = pd.read_csv(custom_file_path, encoding='utf-8-sig')
    
    # 4. 检查是否包含标记列 'is_stopword'
    if 'is_stopword' in df.columns:
        # 提取被标记为 1 的无意义词汇
        condition = df['is_stopword'] == 1
        new_stopwords = df[condition]['word'].dropna().astype(str).tolist()
        
        # 5. 合并停用词
        final_stopwords.update(new_stopwords)
        print(f"成功加载 {len(new_stopwords)} 个自定义停用词。")
    else:
        print("提示：CSV 文件中未找到 'is_stopword' 标记列，请确认是否已添加。")

    return final_stopwords
def setup_custom_dictionary(dict_path='nikki_dict.txt'):
    """
    加载自定义游戏专有名词字典。
    必须在所有分词操作开始前调用此函数。
    """
    if os.path.exists(dict_path):
        jieba.load_userdict(dict_path)
        print(f"成功加载自定义字典：{dict_path}")
    else:
        print(f"提示：未找到字典文件 {dict_path}，将使用默认分词模式。")
def tokenize_zh_factory(stopwords):
    """
    生成带有停用词过滤和词性筛选的分词函数。
    """
    def tokenize(text):
        text = str(text)
        
        # 使用 jieba.posseg.cut 进行分词，它会同时返回词汇和词性
        words_with_flags = pseg.cut(text)
        
        result = []
        for w in words_with_flags:
            word = w.word
            flag = w.flag
            
            # 基础过滤：去除首尾空格、去除空字符串
            clean_word = word.strip()
            
            # 组合过滤条件：
            # 1. 词不能为空
            # 2. 词不能在停用词表 stopwords 中
            if clean_word != '' and clean_word not in stopwords:
                # 【可选】词性过滤：如果需要提升主题建模质量，可以仅保留特定词性。
                # 例如：n(名词), nz(专有名词), vn(名动词), v(动词)
                # 如果暂时不需要词性过滤，可以去掉下面这个 if 判断，直接执行 append
                if flag.startswith('n') or flag.startswith('v'):
                    result.append(clean_word)
                    
        return result
    
    return tokenize

In [ ]:

class TextAnalyzer:
    def __init__(self):
        # 初始化你的停用词
        self.stopwords = set(NIKKI_STOPWORDS)
        self.vectorizer = None
        self.words = None
        

    def process_and_vectorize(self, df1_final, min_df=1):
        """
        假设 df1_final 是之前跑完 15 分钟窗口逻辑后生成的 DataFrame，
        其中包含 'text_concat' 列。
        """
        # 1. 提取所有非空的合并文本，作为 documents_per_class
        # 过滤掉空字符串，避免 CountVectorizer 处理空列表报错
        valid_docs_series = df1_final[df1_final['text_concat'].str.strip() != '']['text_concat']
        documents_per_class = valid_docs_series.tolist()
        
        # 如果没有有效文本，提前返回
        if not documents_per_class:
            return None, []
        setup_custom_dictionary('nikki_dict.txt')
        updated_stopwords = load_dynamic_stopwords('vocabulary_frequencies.csv')

        # 2. 实例化 Vectorizer (严格按照你提供的逻辑)
        self.vectorizer = CountVectorizer(
            tokenizer=tokenize_zh_factory(updated_stopwords), 
            min_df=min_df,
            token_pattern=r"(?u)\b\w+\b" 
        )
        
        # 3. 执行向量化转化
        # X 是一个稀疏矩阵 (Sparse Matrix)，包含每个窗口的词频分布
        X = self.vectorizer.fit_transform(documents_per_class)
        
        # 4. 获取特征词表
        self.words = self.vectorizer.get_feature_names_out()
        
        # 可以选择将生成的稀疏矩阵或词表返回，供后续主题建模使用
        return X, self.words


    def extract_and_export_words(X, words, output_filename='vocabulary_frequencies.csv'):
        """
        根据 CountVectorizer 生成的矩阵和词表，提取所有词汇及其词频，并导出为 CSV 文件。
        
        参数:
        X: CountVectorizer.fit_transform 生成的稀疏矩阵
        words: CountVectorizer.get_feature_names_out() 生成的词汇表
        output_filename: 导出的文件名
        """
        # 1. 计算每个词的总词频
        # X.sum(axis=0) 会将所有文档（行）中的词频按词（列）相加
        # 转换为 numpy array 并使用 flatten() 展平为一维数组
        word_frequencies = np.array(X.sum(axis=0)).flatten()
        
        # 2. 将词汇和对应的词频组合成 DataFrame
        df_vocab = pd.DataFrame({
            'word': words,
            'frequency': word_frequencies
        })
        
        # 3. 按照词频从高到低进行排序
        # 高频词通常是最需要优先检查是否应当作为停用词过滤掉的目标
        df_vocab_sorted = df_vocab.sort_values(by='frequency', ascending=False).reset_index(drop=True)
        
        # 4. 导出为 CSV 文件
        # 使用 'utf-8-sig' 编码确保在 Excel 中打开时中文字符不会乱码
        df_vocab_sorted.to_csv(output_filename, index=False, encoding='utf-8-sig')
        
        return df_vocab_sorted


## 输出分词

In [ ]:
df_reasoning.columns

In [ ]:
df_reasoning[['timestamp','is_baseline_drift',
       'ap_trigger_count', 'pred_label', 'attribution_time', 'dominant_model',
       'dominant_signal', 'trigger_features', 'baseline_version', 'state',
       'text_concat', 'top_img_phash', 'top_img_count', 'top_img_ratio']].to_csv('df_reasoning.csv', index=False, encoding='utf-8-sig')

In [ ]:
analyzer = TextAnalyzer()
X_matrix, feature_words = analyzer.process_and_vectorize(df_reasoning , min_df=2)
df_all_words = analyzer.extract_and_export_words(X_matrix, feature_words)
df_all_words.to_csv('vocabulary_frequencies.csv', index=False, encoding='utf-8-sig')

# 归因

In [ ]:
from TFT_Attribution_engine import ContentAttributionEngine

# 论文用

## 6.3 事件池标记

In [ ]:
import pandas as pd
import numpy as np

def map_crisis_lifecycle_to_reasoning(df_reasoning, crisis_df, time_col='attribution_time'):
    """
    将危机事件池的生命周期阶段映射到 df_reasoning 中。
    
    参数:
    df_reasoning: 包含异常归因结果的 DataFrame
    crisis_df: 包含危机事件生命周期时间戳的 DataFrame
    time_col: df_reasoning 中用于匹配的时间列名，默认为 'attribution_time'
    """
    # 1. 确保时间列都是标准的 datetime 格式
    df_reasoning[time_col] = pd.to_datetime(df_reasoning[time_col], errors='coerce')
    crisis_df['t_crisis_start'] = pd.to_datetime(crisis_df['t_crisis_start'])
    crisis_df['t_crisis_peak'] = pd.to_datetime(crisis_df['t_crisis_peak'])
    crisis_df['t_official_resp'] = pd.to_datetime(crisis_df['t_official_resp'])
    crisis_df['t_crisis_end'] = pd.to_datetime(crisis_df['t_crisis_end'])

    # 2. 初始化新增的列
    df_reasoning['in_crisis'] = False
    df_reasoning['crisis_id'] = None
    df_reasoning['crisis_level'] = None
    df_reasoning['crisis_phase'] = '非危机时期' # 默认状态

    # 3. 遍历危机事件池，使用掩码进行高效匹配和赋值
    for _, crisis in crisis_df.iterrows():
        t_start = crisis['t_crisis_start']
        t_peak = crisis['t_crisis_peak']
        t_resp = crisis['t_official_resp']
        t_end = crisis['t_crisis_end']
        
        c_id = crisis.get('crisis_id')
        c_lvl = crisis.get('crisis_level')

        # 找到整体落在该危机事件时间窗口内的数据
        mask_all = (df_reasoning[time_col] >= t_start) & (df_reasoning[time_col] <= t_end)
        
        if not mask_all.any():
            continue

        # 赋予基础危机属性
        df_reasoning.loc[mask_all, 'in_crisis'] = True
        df_reasoning.loc[mask_all, 'crisis_id'] = c_id
        df_reasoning.loc[mask_all, 'crisis_level'] = c_lvl

        # 根据 Gantt 图的逻辑切分三个阶段
        # 酝酿期: [t_start, t_peak)
        mask_pre = mask_all & (df_reasoning[time_col] < t_peak)
        # 爆发期: [t_peak, t_resp)
        mask_mid = mask_all & (df_reasoning[time_col] >= t_peak) & (df_reasoning[time_col] < t_resp)
        # 消退期: [t_resp, t_end]
        mask_post = mask_all & (df_reasoning[time_col] >= t_resp)

        # 赋予阶段标签
        df_reasoning.loc[mask_pre, 'crisis_phase'] = '酝酿期'
        df_reasoning.loc[mask_mid, 'crisis_phase'] = '爆发期'
        df_reasoning.loc[mask_post, 'crisis_phase'] = '消退期'

    return df_reasoning

# 使用示例：
# 假设你的归因时间列名是 'attribution_time'

# 查看匹配结果统计

In [ ]:
import pandas as pd

def add_events_and_calculate_distribution(df_reasoning, time_col='attribution_time'):
    """
    1. 为 df_reasoning 增加连续事件段的编号 (event_id)
    2. 统计 PA/CA 在各个危机阶段的预警次数与事件数及比例
    """
    # ============================================================
    # 1. 判断事件段并按时间顺序编号
    # ============================================================
    # 确保数据按时间排序，以保证 shift(1) 逻辑正确
    df_reasoning = df_reasoning.sort_values(time_col).reset_index(drop=True)
    
    # 判断逻辑：如果前一条数据的 pred_label 和当前不同，或者时间间隔大于 15 分钟(一个标准窗口)，则视为新事件
    time_diff = df_reasoning[time_col] - df_reasoning[time_col].shift(1)
    label_diff = df_reasoning['pred_label'] != df_reasoning['pred_label'].shift(1)
    
    # 标记新事件的起点
    is_new_event = label_diff | (time_diff > pd.Timedelta(minutes=15))
    
    # 累加生成全局递增的事件编号 (从 1 开始)
    df_reasoning['event_id'] = is_new_event.cumsum()
    
    # ============================================================
    # 2. 统计各个危机阶段的分布比例
    # ============================================================
    # 过滤掉未知标签（如果有的话），只统计我们要看的 PA 和 CA
    valid_mask = df_reasoning['pred_label'].isin(['PA', 'CA'])
    df_valid = df_reasoning[valid_mask].copy()
    
    # 按阶段和特征类型分组统计
    stats = df_valid.groupby(['pred_label', 'crisis_phase']).agg(
        warning_count=('event_id', 'count'),  # 行数 = 预警次数 (点数)
        event_count=('event_id', 'nunique')   # 独立的 event_id 数量 = 事件数 (段数)
    ).reset_index()

    # 计算各标签 (PA/CA) 在总预警/总事件中的占比 (纵向占比计算)
    total_warnings = stats.groupby('pred_label')['warning_count'].transform('sum')
    total_events = stats.groupby('pred_label')['event_count'].transform('sum')
    
    stats['warning_ratio'] = (stats['warning_count'] / total_warnings).apply(lambda x: f"{x:.1%}")
    stats['event_ratio'] = (stats['event_count'] / total_events).apply(lambda x: f"{x:.1%}")
    
    # 按照特定顺序对阶段进行排序，方便阅读
    phase_order = {'酝酿期': 1, '爆发期': 2, '消退期': 3, '非危机时期': 4}
    stats['phase_order'] = stats['crisis_phase'].map(phase_order)
    stats = stats.sort_values(['pred_label', 'phase_order']).drop(columns=['phase_order'])
    
    return df_reasoning, stats

def print_distribution_report(stats_df):
    """
    打印排版精美的统计报表
    """
    print("\n" + "="*65)
    print(" 预警特征 (PA/CA) 在危机生命周期的分布统计")
    print("="*65)
    
    for label in ['PA', 'CA']:
        label_stats = stats_df[stats_df['pred_label'] == label]
        if label_stats.empty:
            continue
            
        print(f"\n[{label} 检测分布]")
        for _, row in label_stats.iterrows():
            phase = row['crisis_phase']
            w_cnt = row['warning_count']
            w_rat = row['warning_ratio']
            e_cnt = row['event_count']
            e_rat = row['event_ratio']
            
            # 使用制表位格式化输出，保持对齐
            print(f"  - {phase:<6}: "
                  f"事件段数 = {e_cnt:<3} ({e_rat:>5})  |  "
                  f"触发次数 = {w_cnt:<4} ({w_rat:>5})")

# 使用示例：
# print_distribution_report(phase_stats)

In [ ]:
crisis_warnings = df_reasoning[df_reasoning['crisis_id'] == 15].copy()
crisis_warnings[(crisis_warnings['is_baseline_drift'] == True) & (crisis_warnings['pred_label'] == 'N')][['pred_label','is_baseline_drift']]['pred_label']= 'CP'
# crisis_warnings[(crisis_warnings['is_baseline_drift'] == True) & (crisis_warnings['pred_label'] == 'N')]['pred_label'] = 'CP'
crisis_warnings[['pred_label','is_baseline_drift']]


In [ ]:
import pandas as pd
import numpy as np

def analyze_crisis_pool_warnings(df_reasoning, crisis_df, time_col='attribution_time'):
    """
    从危机事件池视角，统计每个危机事件的预警命中情况。
    """
    # 确保时间格式正确
    df_reasoning[time_col] = pd.to_datetime(df_reasoning[time_col], errors='coerce')
    crisis_df['t_official_resp'] = pd.to_datetime(crisis_df['t_official_resp'], errors='coerce')
    
    results = []

    for _, crisis in crisis_df.iterrows():
        c_id = crisis['crisis_id']
        t_start = crisis['t_crisis_start']
        t_peak = crisis['t_crisis_peak']
        t_end = crisis['t_crisis_end']
        t_resp = crisis['t_official_resp']
        c_level = crisis.get('crisis_level', 'Unknown')
        
        # 提取当前危机事件关联的所有归因记录
        crisis_warnings = df_reasoning[df_reasoning['crisis_id'] == c_id].copy()
        # print(crisis_warnings)
        mask = (crisis_warnings['is_baseline_drift'] == True) & (crisis_warnings['pred_label'] == 'N')
        crisis_warnings.loc[mask, 'pred_label'] = 'CP'
        # 仅保留有效的预警类型 PA 和 CA
        mask = crisis_warnings['pred_label'].isin(['PA', 'CA', 'CP'])
        # crisis_warnings = crisis_warnings[crisis_warnings['pred_label'].isin(['PA', 'CA'])]
        crisis_warnings = crisis_warnings[mask]

        
        # 如果该危机没有匹配到任何预警记录
        if crisis_warnings.empty:
            results.append({
                'crisis_id': c_id,
                'crisis_level': c_level,
                't_crisis_start': t_start,
                't_crisis_peak': t_peak,
                't_crisis_end': t_end,
                't_official_resp': t_resp,
                'first_warning_time': pd.NaT,
                'lead_time_minutes': np.nan,  # 提前时间
                'first_warning_type': '无预警',
                'PA_event_count': 0, 'PA_warning_count': 0,
                'CA_event_count': 0, 'CA_warning_count': 0,
                'CP_event_count': 0, 'CP_warning_count': 0,
            })
            continue
            
        # 按照时间排序，获取第一条预警记录
        crisis_warnings = crisis_warnings.sort_values(time_col)
        first_warning = crisis_warnings.iloc[0]
        
        first_w_time = first_warning[time_col]
        first_w_type = first_warning['pred_label']
        
        # 计算相比官方响应的提前时间 (分钟)。正数代表提前，负数代表延后。
        lead_time = np.nan
        if pd.notna(t_resp) and pd.notna(first_w_time):
            lead_time = (t_resp - first_w_time).total_seconds() / 60.0
            
        # 统计分类型的预警次数(行数)和事件段数量(去重的event_id数)
        stats = crisis_warnings.groupby('pred_label').agg(
            w_count=(time_col, 'count'),
            e_count=('event_id', 'nunique')
        ).to_dict(orient='index')
        # stats_drift = crisis_warnings[crisis_warnings['is_baseline_drift'] == True].groupby('is_baseline_drift').agg(
        #     w_count=(time_col, 'count'),
        #     e_count=('event_id', 'nunique')
        # ).to_dict(orient='index')
        
        pa_stats = stats.get('PA', {'w_count': 0, 'e_count': 0})
        ca_stats = stats.get('CA', {'w_count': 0, 'e_count': 0})
        cp_stats = stats.get('CP', {'w_count': 0, 'e_count': 0})

        results.append({
            'crisis_id': c_id,
            'crisis_level': c_level,
            't_crisis_start': t_start,
            't_crisis_peak': t_peak,
            't_crisis_end': t_end,
            't_official_resp': t_resp,
            'first_warning_time': first_w_time,
            'lead_time_minutes': lead_time,
            'first_warning_type': first_w_type,
            'PA_event_count': pa_stats['e_count'],
            'PA_warning_count': pa_stats['w_count'],
            'CA_event_count': ca_stats['e_count'],
            'CA_warning_count': ca_stats['w_count'],
            'CP_event_count': cp_stats['e_count'],
            'CP_warning_count': cp_stats['w_count']
        })
        
    return pd.DataFrame(results)

def print_crisis_pool_report(pool_stats_df):
    """
    打印危机事件池预警统计报表
    """
    print("\n" + "="*85)
    print(" 危机事件池预警命中情况统计 (提前时间正数为提前，负数为延后)")
    print("="*85)
    
    # 格式化输出表头
    header = f"{'Crisis ID':<10} | {'Level':<5} | {'首个预警时间':<20} | {'首发类型':<8} | {'提前(分钟)':<10} | {'PA(段/次)':<10} | {'CA(段/次)':<10}| {'CP(段/次)':<10}"
    print(header)
    print("-" * 85)
    
    for _, row in pool_stats_df.iterrows():
        c_id = str(row['crisis_id'])
        c_lvl = str(row['crisis_level'])
        
        first_time = row['first_warning_time'].strftime('%m-%d %H:%M') if pd.notna(row['first_warning_time']) else "未检出"
        w_type = row['first_warning_type']
        
        # 格式化提前时间
        lead_time = row['lead_time_minutes']
        lead_str = f"{lead_time:+.1f}" if pd.notna(lead_time) else "N/A"
        
        pa_info = f"{row['PA_event_count']}/{row['PA_warning_count']}"
        ca_info = f"{row['CA_event_count']}/{row['CA_warning_count']}"
        cp_info = f"{row['CP_event_count']}/{row['CP_warning_count']}"
        
        print(f"{c_id:<10} | {c_lvl:<5} | {first_time:<20} | {w_type:<8} | {lead_str:<10} | {pa_info:<10} | {ca_info:<10}| {cp_info:<10}")

# 使用示例：


### 用attribution time标记

In [ ]:
df_reasoning_mapped = map_crisis_lifecycle_to_reasoning(df_reasoning, crisis_df, time_col='attribution_time')
df_reasoning_added, phase_stats = add_events_and_calculate_distribution(df_reasoning_mapped, time_col='attribution_time')



In [ ]:
df_reasoning_added.columns


In [ ]:
df_reasoning_added[['time_idx', 'timestamp',  'is_baseline_drift',
       'ap_trigger_count', 'pred_label', 'attribution_time', 'dominant_model',
       'dominant_signal', 'trigger_features', 'baseline_version', 'state', 'in_crisis', 'crisis_id', 'crisis_level', 'crisis_phase', 'event_id']].to_csv('df_reasoning_with_crisis_phases.csv', index=False, encoding='utf-8-sig')

In [ ]:
crisis_df=crisis_df[crisis_df['t_crisis_start']>pd.to_datetime('2025-03-01')]


In [ ]:
print(crisis_pool_stats)

In [ ]:
crisis_pool_stats = analyze_crisis_pool_warnings(df_reasoning_added, crisis_df[crisis_df['t_crisis_start']>pd.to_datetime('2025-03-01')], time_col='attribution_time')
crisis_pool_stats
print_crisis_pool_report(crisis_pool_stats)

### 用timestamp标记

In [ ]:
df_reasoning_mapped = map_crisis_lifecycle_to_reasoning(df_reasoning, crisis_df, time_col='timestamp')
df_reasoning_added, phase_stats = add_events_and_calculate_distribution(df_reasoning_mapped, time_col='timestamp')


In [ ]:
df_reasoning_added[['time_idx', 'timestamp',  'is_baseline_drift',
       'ap_trigger_count', 'pred_label', 'attribution_time', 'dominant_model',
       'dominant_signal', 'trigger_features', 'baseline_version', 'state', 'in_crisis', 'crisis_id', 'crisis_level', 'crisis_phase', 'event_id']].to_csv('df_reasoning_with_crisis_phases_v2.csv', index=False, encoding='utf-8-sig')


In [ ]:
crisis_pool_stats = analyze_crisis_pool_warnings(df_reasoning_added, crisis_df[crisis_df['t_crisis_start']>pd.to_datetime('2025-03-01')], time_col='attribution_time')
crisis_pool_stats
print_crisis_pool_report(crisis_pool_stats)

In [ ]:
crisis_pool_stats.info()

## 危机等级图

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 设置中文字体，防止图表中的中文显示为方块 (根据你的操作系统选择合适的字体)
plt.rcParams['font.sans-serif'] = ['SimHei']  # Windows系统常用黑体
# plt.rcParams['font.sans-serif'] = ['Arial Unicode MS'] # Mac系统常用
plt.rcParams['axes.unicode_minus'] = False 

def plot_crisis_warning_stacked_bar(crisis_pool_stats):
    """
    绘制危机等级与最高预警层级的堆叠柱状图
    """
    df = crisis_pool_stats.copy()
    
    # 1. 确定每次危机的最高预警级别
    def determine_highest_warning(row):
        if row['CA_warning_count'] > 0  or row['CP_warning_count'] > 0:
            return '最高触发 CA/CP'
        elif row['PA_warning_count'] > 0:
            return '最高触发 PA'
        else:
            return '未检测'
            
    df['highest_warning'] = df.apply(determine_highest_warning, axis=1)
    
    # 2. 聚合数据：按危机等级和最高预警级别进行计数
    plot_data = df.groupby(['crisis_level', 'highest_warning']).size().unstack(fill_value=0)
    
    # 确保所有类别列都存在，并且按我们期望的颜色堆叠顺序排列 (从下到上)
    categories = ['未检测', '最高触发 PA', '最高触发 CA/CP']
    for cat in categories:
        if cat not in plot_data.columns:
            plot_data[cat] = 0
            
    # 重排序列，保证画图时的堆叠顺序一致
    plot_data = plot_data[categories]
    
    # 修改横坐标标签为 Level 1, Level 2...
    plot_data.index = [f'Level {int(idx)}' for idx in plot_data.index]
    
    # 3. 开始绘图
    fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
    
    # 定义颜色：灰色(漏警) -> 浅珊瑚色/橙色(PA) -> 砖红色(CA)
    colors = ['#A9A9A9', '#FFA07A', '#B22222'] 
    
    # 画堆叠柱状图
    plot_data.plot(
        kind='bar', 
        stacked=True, 
        color=colors, 
        ax=ax, 
        edgecolor='black', # 增加黑色边框让柱体更立体
        width=0.5,         # 控制柱体宽度
        zorder=3           # 让柱体在网格线之上
    )
    
    # 图表格式化美化
    # ax.set_title('危机等级与最高预警层级分布', fontsize=15, fontweight='bold', pad=15)
    ax.set_xlabel('真实危机等级', fontsize=12)
    ax.set_ylabel('危机事件数量 (个)', fontsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=11)
    
    # 添加水平网格线，辅助阅读数据
    ax.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)
    
    # 调整图例位置，放在图表右侧避免遮挡柱体
    ax.legend(title='最终响应状态', fontsize=10, title_fontsize=11, 
              loc='upper left', bbox_to_anchor=(1.02, 1))
    
    plt.tight_layout()
    
    # 保存图片，适合插入论文
    save_path = "crisis_warning_stacked_bar.png"
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    print(f"✅ 图表已生成并保存至: {save_path}")
    
    plt.show()

# 调用函数 (假设你的 df 变量名为 crisis_pool_stats)
plot_crisis_warning_stacked_bar(crisis_pool_stats)

## 事件图

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D

def plot_relative_crisis_timeline(crisis_df, df_pic, custom_start, custom_end, output_file):
    # 预设学术排版参数
    plt.rcParams.update({
        'font.size': 14,
        'axes.labelsize': 16,
        'axes.titlesize': 18,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'legend.fontsize': 13
    })

    # 1. 过滤自定义时间范围内的危机事件
    crisis_df['t_crisis_start'] = pd.to_datetime(crisis_df['t_crisis_start'])
    crisis_df['t_crisis_peak'] = pd.to_datetime(crisis_df['t_crisis_peak'])
    crisis_df['t_official_resp'] = pd.to_datetime(crisis_df['t_official_resp'])
    crisis_df['t_crisis_end'] = pd.to_datetime(crisis_df['t_crisis_end'])
    
    mask_crisis = (crisis_df['t_crisis_start'] >= custom_start) & (crisis_df['t_crisis_end'] <= custom_end)
    target_crisis = crisis_df[mask_crisis].copy().reset_index(drop=True)
    
    if target_crisis.empty:
        print("设定时间范围内无符合条件的危机事件。")
        return

    fig, ax = plt.subplots(figsize=(14, max(6, len(target_crisis) * 0.8)))
    
    # 学术配色体系
    colors = {
        'pre': '#AED6F1',        # 酝酿期
        'mid': '#F5B7B1',        # 爆发期
        'post': '#E5E7E9',       # 消退期
        'pa': '#F39C12',         # PA 散点
        'ca': '#E74C3C',         # CA 线段
        'cp': '#8E44AD'          # CP 线段
    }

    y_labels = []
    
    # 2. 遍历事件并计算相对时间（单位：小时）
    for i, row in target_crisis.iterrows():
        y_pos = i
        y_labels.append(f"事件 {row['crisis_id']-5}\n(Lvl {row['crisis_level']})")
        
        t_p = row['t_crisis_peak']
        
        # 相对时间转换
        rel_start = (row['t_crisis_start'] - t_p).total_seconds() / 3600.0
        rel_peak = 0.0
        rel_resp = (row['t_official_resp'] - t_p).total_seconds() / 3600.0
        rel_end = (row['t_crisis_end'] - t_p).total_seconds() / 3600.0
        
        # 绘制生命周期背景色块
        ax.barh(y_pos, 0 - rel_start, left=rel_start, color=colors['pre'], alpha=0.6, edgecolor='none')
        ax.barh(y_pos, rel_resp - 0, left=0, color=colors['mid'], alpha=0.6, edgecolor='none')
        ax.barh(y_pos, rel_end - rel_resp, left=rel_resp, color=colors['post'], alpha=0.6, edgecolor='none')
        
        # 绘制关键节点的相对时间参考线
        ax.vlines(0, ymin=y_pos-0.4, ymax=y_pos+0.4, color='red', linestyle='--', lw=2, alpha=0.8)
        
        # 3. 提取对应区间的预警信号并转换为相对时间
        mask_sig = (df_pic['timestamp'] >= row['t_crisis_start']) & (df_pic['timestamp'] <= row['t_crisis_end'])
        sub_df = df_pic[mask_sig].copy()
        
        # 修正：补充 .dt 访问器以支持 Series 的向量化操作
        sub_df['rel_time'] = (pd.to_datetime(sub_df['timestamp']) - t_p).dt.total_seconds() / 3600.0
        
        pa_pts = sub_df[sub_df['pred_label'] == 'PA']
        ca_pts = sub_df[sub_df['pred_label'] == 'CA']
        cp_pts = sub_df[sub_df['is_baseline_drift'] == True] if 'is_baseline_drift' in sub_df.columns else pd.DataFrame()
        
        # 绘制 PA (散点)
        if not pa_pts.empty:
            ax.scatter(pa_pts['rel_time'], [y_pos]*len(pa_pts), color=colors['pa'], marker='o', s=60, zorder=3)
        
        # 绘制 CA (以较宽的矩形标记模拟线段)
        if not ca_pts.empty:
            ax.scatter(ca_pts['rel_time'], [y_pos]*len(ca_pts), color=colors['ca'], marker='|', s=300, lw=3, zorder=4)
            
        # 绘制 CP (以较宽的矩形标记模拟线段)
        if not cp_pts.empty:
            ax.scatter(cp_pts['rel_time'], [y_pos]*len(cp_pts), color=colors['cp'], marker='|', s=300, lw=3, zorder=5)

    # 4. 坐标轴与排版设置
    ax.set_yticks(range(len(target_crisis)))
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("相对时间 (小时) [$t_{peak}=0$]")
    ax.set_title("图 6.5 典型危机事件预警信号的生命周期时序映射", pad=20)
    
    # 强制在 X=0 处绘制一条全局垂直虚线作为视觉基准
    ax.axvline(0, color='gray', linestyle='-.', lw=1, alpha=0.5)

    # 5. 图例构造
    proxy_pre = plt.Rectangle((0,0),1,1, fc=colors['pre'], alpha=0.6, label='酝酿期 (Pre)')
    proxy_mid = plt.Rectangle((0,0),1,1, fc=colors['mid'], alpha=0.6, label='爆发期 (Mid)')
    proxy_post = plt.Rectangle((0,0),1,1, fc=colors['post'], alpha=0.6, label='消退期 (Post)')
    proxy_peak = Line2D([0], [0], color='red', linestyle='--', lw=2, label='峰值原点 ($t_{peak}$)')
    proxy_pa = Line2D([0], [0], marker='o', color='w', markerfacecolor=colors['pa'], markersize=9, label='PA 信号 (散点)')
    proxy_ca = Line2D([0], [0], marker='|', color='w', markeredgecolor=colors['ca'], markersize=14, markeredgewidth=3, label='CA 信号 (线段)')
    proxy_cp = Line2D([0], [0], marker='|', color='w', markeredgecolor=colors['cp'], markersize=14, markeredgewidth=3, label='CP 信号 (线段)')

    legend_handles = [proxy_pre, proxy_mid, proxy_post, proxy_peak, proxy_pa, proxy_ca, proxy_cp]
    
    # 图例水平三列排布于上方
    ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4, framealpha=0.95, edgecolor='black')

    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.tight_layout()

    plt.savefig(output_file, dpi=300, bbox_inches='tight', format='svg')
    plt.show()
    print(f"图表渲染完成，已导出至: {output_file}")

In [ ]:
df_pic

In [ ]:
# 执行示例 (需根据实际数据的日期范围替换时间参数)
output_file = './output/relative_crisis_timeline_2.svg'
start_time = pd.to_datetime('2025-04-25')
end_time = pd.to_datetime('2025-05-25')
plot_relative_crisis_timeline(crisis_df, df_reasoning, start_time, end_time,output_file)

In [ ]:
# 执行示例 (需根据实际数据的日期范围替换时间参数)
output_file = './output/relative_crisis_timeline_7_8.svg'
start_time = pd.to_datetime('2025-08-25')
end_time = pd.to_datetime('2025-09-15')
plot_relative_crisis_timeline(crisis_df, df_reasoning, start_time, end_time,output_file)

In [ ]:
# 执行示例 (需根据实际数据的日期范围替换时间参数)
output_file = './output/relative_crisis_timeline_10.svg'
start_time = pd.to_datetime('2025-10-25')
end_time = pd.to_datetime('2025-11-15')
plot_relative_crisis_timeline(crisis_df, df_reasoning, start_time, end_time,output_file)

## 事件图V2

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

def plot_relative_crisis_timeline(crisis_df, df_pic, custom_start, custom_end, output_file):
    plt.rcParams.update({
        'font.size': 14,
        'axes.labelsize': 16,
        'axes.titlesize': 18,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'legend.fontsize': 13
    })

    crisis_df['t_crisis_start'] = pd.to_datetime(crisis_df['t_crisis_start'])
    crisis_df['t_crisis_peak'] = pd.to_datetime(crisis_df['t_crisis_peak'])
    crisis_df['t_official_resp'] = pd.to_datetime(crisis_df['t_official_resp'])
    crisis_df['t_crisis_end'] = pd.to_datetime(crisis_df['t_crisis_end'])
    
    mask_crisis = (crisis_df['t_crisis_start'] >= custom_start) & (crisis_df['t_crisis_end'] <= custom_end)
    target_crisis = crisis_df[mask_crisis].copy().reset_index(drop=True)
    
    if target_crisis.empty:
        print("设定时间范围内无符合条件的危机事件。")
        return

    fig, ax = plt.subplots(figsize=(14, max(6, len(target_crisis) * 0.8)))
    
    colors = {
        'pre': '#AED6F1',        
        'mid': '#F5B7B1',        
        'post': '#E5E7E9',       
        'pa': '#F39C12',         
        'ca': '#E74C3C',         
        'cp': '#8E44AD'          
    }

    y_labels = []
    window_hr = 15.0 / 60.0
    
    for i, row in target_crisis.iterrows():
        y_pos = i
        y_labels.append(f"事件 {row['crisis_id']-5}\n(Lvl {row['crisis_level']})")
        
        t_p = row['t_crisis_peak']
        
        rel_start = (row['t_crisis_start'] - t_p).total_seconds() / 3600.0
        rel_resp = (row['t_official_resp'] - t_p).total_seconds() / 3600.0
        rel_end = (row['t_crisis_end'] - t_p).total_seconds() / 3600.0
        
        # 背景色块 (zorder=1)
        ax.barh(y_pos, 0 - rel_start, left=rel_start, color=colors['pre'], alpha=0.6, edgecolor='none', zorder=1)
        ax.barh(y_pos, rel_resp - 0, left=0, color=colors['mid'], alpha=0.6, edgecolor='none', zorder=1)
        ax.barh(y_pos, rel_end - rel_resp, left=rel_resp, color=colors['post'], alpha=0.6, edgecolor='none', zorder=1)
        
        mask_sig = (df_pic['timestamp'] >= row['t_crisis_start']) & (df_pic['timestamp'] <= row['t_crisis_end'])
        sub_df = df_pic[mask_sig].copy()
        sub_df['rel_time'] = (pd.to_datetime(sub_df['timestamp']) - t_p).dt.total_seconds() / 3600.0
        
        pa_pts = sub_df[sub_df['pred_label'].isin(['PA', 'AP'])]
        ca_pts = sub_df[sub_df['pred_label'].isin(['CA', 'CP'])]
        
        # 变点信号分类逻辑
        cp_solid = sub_df[(sub_df['is_baseline_drift'] == True) & (sub_df['pred_label'] == 'N')]
        cp_trans = sub_df[(sub_df['is_baseline_drift'] == True) & (sub_df['pred_label'] != 'N')]
        
        # 预警信号 (zorder=2，统一缩减高度)
        if not pa_pts.empty:
            ax.barh(y=[y_pos]*len(pa_pts), width=window_hr, left=pa_pts['rel_time'] - window_hr, 
                    color=colors['pa'], height=0.15, zorder=2)
        
        if not ca_pts.empty:
            ax.barh(y=[y_pos]*len(ca_pts), width=window_hr, left=ca_pts['rel_time'] - window_hr, 
                    color=colors['ca'], height=0.3, zorder=2)
            
        if not cp_solid.empty:
            ax.barh(y=[y_pos]*len(cp_solid), width=window_hr, left=cp_solid['rel_time'] - window_hr, 
                    color=colors['cp'], height=0.45, zorder=2)

        if not cp_trans.empty:
            ax.barh(y=[y_pos]*len(cp_trans), width=window_hr, left=cp_trans['rel_time'] - window_hr, 
                    color=colors['cp'], height=0.45, alpha=0.3, zorder=2)

        # 核心参考时间线 (zorder=3，置于最顶层)
        ax.vlines(0, ymin=y_pos-0.4, ymax=y_pos+0.4, color='red', linestyle='--', lw=1.5, zorder=3)
        ax.vlines(rel_resp, ymin=y_pos-0.4, ymax=y_pos+0.4, color='black', linestyle='-', lw=1.5, zorder=3)

    ax.set_yticks(range(len(target_crisis)))
    ax.set_yticklabels(y_labels)
    ax.set_xlabel(r"相对时间 (小时) [$t_{peak}=0$]")
    ax.set_title("图 6.5 典型危机事件预警信号的生命周期时序映射", pad=20)
    
    ax.axvline(0, color='gray', linestyle='-.', lw=1, alpha=0.5, zorder=0)

    # 构造综合图例
    proxy_pre = mpatches.Patch(color=colors['pre'], alpha=0.6, label='酝酿期 (Pre)')
    proxy_mid = mpatches.Patch(color=colors['mid'], alpha=0.6, label='爆发期 (Mid)')
    proxy_post = mpatches.Patch(color=colors['post'], alpha=0.6, label='消退期 (Post)')
    
    proxy_peak = Line2D([0], [0], color='red', linestyle='--', lw=1.5, label=r'峰值原点 ($t_{peak}$)')
    proxy_resp = Line2D([0], [0], color='black', linestyle='-', lw=1.5, label='官方响应时间')
    
    proxy_pa = mpatches.Patch(color=colors['pa'], label='PA 信号 (15min窗口)')
    proxy_ca = mpatches.Patch(color=colors['ca'], label='CA 信号 (15min窗口)')
    proxy_cp_solid = mpatches.Patch(color=colors['cp'], label='CP 信号 (15min窗口)')
    # proxy_cp_trans = mpatches.Patch(color=colors['cp'], alpha=0.3, label='CP 信号 (并发复合)')
    # proxy_cp_trans = mpatches.Patch(color=colors['cp'], alpha=0.3)


    legend_handles = [
        proxy_pre, proxy_mid, proxy_post, 
        proxy_peak, proxy_resp, proxy_pa, 
        proxy_ca, proxy_cp_solid
        # , proxy_cp_trans
    ]
    
    ax.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, framealpha=0.95, edgecolor='black')

    plt.grid(axis='x', linestyle='--', alpha=0.4, zorder=0)
    plt.tight_layout()

    plt.savefig(output_file, dpi=300, bbox_inches='tight', format='svg')
    plt.show()
    print(f"图表渲染完成，已导出至: {output_file}")

# 执行示例


In [ ]:
output_file = './output/relative_crisis_timeline_v2.2.svg'
start_time = pd.to_datetime('2025-04-27')
end_time = pd.to_datetime('2025-05-15')
plot_relative_crisis_timeline(crisis_df, df_detected, start_time, end_time, output_file)

In [ ]:
output_file = './output/relative_crisis_timeline_v2.svg'
start_time = pd.to_datetime('2025-04-01')
end_time = pd.to_datetime('2025-11-15')
plot_relative_crisis_timeline(crisis_df, df_detected, start_time, end_time, output_file)

In [ ]:
output_file = './output/relative_crisis_timeline_v2.10.svg'
start_time = pd.to_datetime('2025-10-25')
end_time = pd.to_datetime('2025-11-26')
plot_relative_crisis_timeline(crisis_df, df_detected, start_time, end_time, output_file)

# end

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 设定随机种子以确保图形可复现
np.random.seed(42)

# 构建时间轴
t = np.arange(0, 300)

# 1. 初始分布 P_1 (常态舆情基线)
y1 = np.random.normal(0, 1, 100)
# 在 t=50 处植入点异常 (孤立的脉冲，随后自发回归分布 P_1)
y1[50] = 6

# 2. 均值漂移分布 P_2 (变点1后，舆情热度中枢永久性抬升)
y2 = np.random.normal(6, 1, 100)

# 3. 方差漂移分布 P_3 (变点2后，舆情波动性永久性加剧)
y3 = np.random.normal(6, 5, 100)

# 拼接为完整时间序列
y = np.concatenate([y1, y2, y3])

# 绘图设置
plt.figure(figsize=(10, 4))
plt.plot(t, y, color='black', linewidth=1.2)

# 绘制点异常标注
# plt.annotate('Point Anomaly\n(Recoverable)', xy=(50, 6), xytext=(15, 7),
plt.annotate('点异常\n(可恢复)', xy=(50, 6), xytext=(15, 7),
             
             arrowprops=dict(facecolor='red', shrink=0.05, width=1, headwidth=5),
             fontsize=15, color='red')

# 绘制变点1标注及其分割线
plt.axvline(x=100, color='blue', linestyle='--', alpha=0.8)
plt.annotate('变点1\n(均值漂移)', xy=(100, 10), xytext=(105, 10),
             fontsize=15, color='blue')

# 绘制变点2标注及其分割线
plt.axvline(x=200, color='green', linestyle='--', alpha=0.8)
plt.annotate('变点2\n(方差漂移)', xy=(200, 20), xytext=(210, 20),
             fontsize=15, color='green')

# 区分不同的分布区间 (P_1, P_2, P_3)
plt.axvspan(0, 100, facecolor='gray', alpha=0.1)
plt.axvspan(100, 200, facecolor='blue', alpha=0.05)
plt.axvspan(200, 300, facecolor='green', alpha=0.05)

# 设置图表属性
# plt.title('Structural Evolution of Public Opinion Metric: Anomalies vs. Change Points', fontsize=12)
plt.xlabel('时间步 (t)', fontsize=10)
# plt.ylabel('Public Opinion Metric', fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.xlim(0, 300)
plt.tight_layout()

# 输出图形
plt.show()